In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer
from datasets import load_dataset, DatasetDict, ClassLabel

import multiprocessing

from functools import partial

from tqdm import tqdm
from collections import Counter

In [ ]:
data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv"
max_token_length = 512
batch_size_train = 64
num_workers = 4
batch_size_val = 64
batch_size_test = 64
val_num_workers = 4
start_idx = 64100
end_idx = 1
padding_idx = 0
unk_idx = 2
seed = 42

In [ ]:
class Loaders():
    def __init__(
            self,
            data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",
            max_token_length = 512,
            batch_size_train = 8,
            num_workers = 4,
            batch_size_val = 4,
            batch_size_test = 4,
            val_num_workers = 4,
            start_idx = 64100, 
            end_idx = 1, 
            padding_idx = 0, 
            unk_idx = 2,
            seed = 42,
    ):
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.padding_idx = padding_idx
        self.unk_idx = unk_idx

        # 1) 데이터셋 로드
        dataset = load_dataset("csv", data_files=data_path)['train']
        # AI hub 한국어-영어 번역(병렬) 말뭉치 데이터셋
        # kor, en, cat 3개의 칼럼으로 구성

        # 2) 카테고리 컬럼의 고유 클래스 찾아서 ClassLabel 객체 생성
        unique_classes = dataset.unique('cat')  
        # 'cat' 열에 문장 분류 
        # 0: 구어체
        # 1: 대화체
        # 2: 문어체_뉴스
        # 3: 문어체_한국문화
        # 4: 문어체_조례
        # 5: 문어체_지자체웹사이트

        print(f"==>> unique_classes: {unique_classes}")
        class_label = ClassLabel(names=unique_classes)

        # 3) 기존 컬럼 타입 변경 (캐스팅)
        dataset = dataset.cast_column('cat', class_label)

        # 4) stratify_by_column으로 분할
        train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='cat')
        valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='cat')

        dataset_dict = DatasetDict({
            'train': train_validtest['train'],
            'validation': valid_test['train'],
            'test': valid_test['test']
        })

        # print(dataset_dict)

        NUM_CPU = multiprocessing.cpu_count()
        # print(f"==>> NUM_CPU: {NUM_CPU}")

        # self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
        # @@@ 점포, 만료 등의 단어가 <unk>인 문제 발견 => 다른 토크나이저 사용?
        self.tokenizer = AutoTokenizer.from_pretrained("KETI-AIR/ke-t5-base")

        special_tokens_dict = {'bos_token': '<s>'}
        self.tokenizer.add_special_tokens(special_tokens_dict)

        print(self.tokenizer.all_special_ids)
        print(self.tokenizer.all_special_tokens)

        print(f"==>> self.tokenizer.model_max_length: {self.tokenizer.model_max_length}")

        print(f"==>> len(self.tokenizer): {len(self.tokenizer)}")

        self.max_token_length = min(max_token_length, self.tokenizer.model_max_length)

        # partial을 이용해 tokenizer, max_token_length 인자 고정
        cetf = partial(convert_examples_to_features, tokenizer=self.tokenizer, max_token_length=self.max_token_length)
        # @@@ convert_examples_to_features함수에서 examples가 첫번쨰 인자가 아니면 
        # @@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서 
        # @@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러 발생

        self.datasets = dataset_dict.map(
                                cetf,
                                # lambda examples: convert_examples_to_features(examples, tokenizer=self.tokenizer, max_token_length=self.max_token_length),
                                batched=True,
                                # 원 데이터 'en', 'kor', 'cat' 등의 칼럼을 지우려면
                                # remove_columns 인자 사용
                                # remove_columns=dataset_dict["train"].column_names,
                                num_proc=NUM_CPU)

        print(f"==>> self.datasets: {self.datasets}")

        self.train_set = self.datasets['train']
        self.val_set = self.datasets['validation']
        self.test_set = self.datasets['test']

        c_fn = partial(collate_fn, start_idx=self.start_idx, end_idx=self.end_idx, padding_idx=self.padding_idx, unk_idx=self.unk_idx)

        self.loader_train = DataLoader(self.train_set, batch_size=batch_size_train, collate_fn=c_fn, shuffle=True, num_workers=num_workers, pin_memory=True)
        # 학습시에만 shuffle=True
        self.loader_val = DataLoader(self.val_set, batch_size=batch_size_val, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)
        self.loader_test = DataLoader(self.test_set, batch_size=batch_size_test, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)

# def convert_examples_to_features(tokenizer, max_token_length, examples):
# @@@@@@@@@ examples가 첫번쨰 인자가 아니면 
# @@@@@@@@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서
# @@@@@@@@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러가 발생한다
def convert_examples_to_features(examples, tokenizer, max_token_length):
    model_inputs = tokenizer(examples['kor'],
                             text_target=examples['en'],
                             max_length=max_token_length, truncation=True)
    # 여기서 첫번째 인자와 두번째 인자의 순서를 바꾸면 한영 번역 대신 영한 번역용으로 인풋과 타겟이 토큰화된다
    return model_inputs


def collate_fn(batch, start_idx, end_idx, padding_idx, unk_idx):
    # print('Original:\n', batch)
    # print("".center(50, "-"))
    # batch는 [{'kor':..., 'en':..., 'cat':숫자, 'input_ids':[...], 'attention_mask':[1, ...], 'labels': [...]}, ...] 형태
    
    # keys = batch[0].keys()
    keys = ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels']
    # print(f"==>> keys: {keys}")
    # print("".center(50, "-"))
    
    # new_batch = {k:[] for k in keys}
    # new_batch['decoder_inputs'] = []
    
    # for b in batch:
    #     for k,v in b.items():
    #         if k == 'input_ids' or k == 'attention_mask':
    #             new_batch[k].append(torch.LongTensor(v))
    #         elif k == 'labels':
    #             new_batch[k].append(torch.LongTensor(v))
    #             new_batch['decoder_inputs'].append(torch.LongTensor([65001] + v[:-1]))
    #         else:
    #             new_batch[k].append(v)

    # key값별로 value 다 모으기
    new_batch = {k:[b[k] for b in batch] for k in keys}

    # list들 LongTensor로 변환
    new_batch['decoder_inputs'] = [torch.LongTensor([start_idx] + label[:-1]) for label in new_batch['labels']]
    # print(f"==>> new_batch['decoder_inputs']: {new_batch['decoder_inputs']}")
    new_batch['input_ids'] = [torch.LongTensor(inp) for inp in new_batch['input_ids']]
    new_batch['labels'] = [torch.LongTensor(label) for label in new_batch['labels']]
    new_batch['attention_mask'] = [torch.LongTensor(mask) for mask in new_batch['attention_mask']]

    new_batch['ntokens'] = sum([l.numel() for l in new_batch['labels']])

    # decoder input의 attention mask 생성
    new_batch['decoder_mask'] = [torch.ones_like(d_inp, dtype=torch.long) for d_inp in new_batch['decoder_inputs']]


    # 각 input과 target 텐서를 패딩
    padded_inputs = pad_sequence(new_batch['input_ids'], batch_first=True, padding_value=padding_idx)
    new_batch['input_ids'] = padded_inputs
    padded_decoder_inputs = pad_sequence(new_batch['decoder_inputs'], batch_first=True, padding_value=padding_idx)
    new_batch['decoder_inputs'] = padded_decoder_inputs
    padded_targets = pad_sequence(new_batch['labels'], batch_first=True, padding_value=padding_idx)
    new_batch['labels'] = padded_targets

    # attention 마스크는 패딩(padding_idx) 대신 False(0)을 입력
    padded_masks = pad_sequence(new_batch['attention_mask'], batch_first=True, padding_value=0)
    new_batch['attention_mask'] = padded_masks

    padded_d_masks = pad_sequence(new_batch['decoder_mask'], batch_first=True, padding_value=0)
    new_batch['decoder_mask'] = padded_d_masks
    
    return new_batch

In [ ]:
loaders = Loaders(
    data_path=data_path,
    max_token_length=max_token_length,
    batch_size_train=batch_size_train,
    num_workers=num_workers,
    batch_size_val=batch_size_val,
    batch_size_test=batch_size_test,
    val_num_workers=val_num_workers,
    start_idx=start_idx,
    end_idx=end_idx,
    padding_idx=padding_idx,
    unk_idx=unk_idx,
    seed=seed,
)

==>> unique_classes: [2, 4, 1, 0, 3, 5]


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


[64100, 1, 2, 0, 64099, 64098, 64097, 64096, 64095, 64094, 64093, 64092, 64091, 64090, 64089, 64088, 64087, 64086, 64085, 64084, 64083, 64082, 64081, 64080, 64079, 64078, 64077, 64076, 64075, 64074, 64073, 64072, 64071, 64070, 64069, 64068, 64067, 64066, 64065, 64064, 64063, 64062, 64061, 64060, 64059, 64058, 64057, 64056, 64055, 64054, 64053, 64052, 64051, 64050, 64049, 64048, 64047, 64046, 64045, 64044, 64043, 64042, 64041, 64040, 64039, 64038, 64037, 64036, 64035, 64034, 64033, 64032, 64031, 64030, 64029, 64028, 64027, 64026, 64025, 64024, 64023, 64022, 64021, 64020, 64019, 64018, 64017, 64016, 64015, 64014, 64013, 64012, 64011, 64010, 64009, 64008, 64007, 64006, 64005, 64004, 64003, 64002, 64001, 64000]
['<s>', '</s>', '<unk>', '<pad>', '<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<ex

In [ ]:
print(f"==>> unk_idx: {unk_idx}")

input_total = Counter([])
target_total = Counter([])
# total

num_batches_train = len(loaders.loader_train)

for step, batch in tqdm(
    enumerate(loaders.loader_train),
    total=num_batches_train,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total += input_count
    target_total += target_count

    # if step == 10:
    #     break

print(f"==>> input_total: {input_total}")
print(f"==>> target_total: {target_total}")

print(f"==>> input_total[unk_idx]: {input_total[unk_idx]}")
print(f"==>> target_total[unk_idx]: {target_total[unk_idx]}")

input_total_val = Counter([])
target_total_val = Counter([])
# total

num_batches_val = len(loaders.loader_val)

for step, batch in tqdm(
    enumerate(loaders.loader_val),
    total=num_batches_val,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_val += input_count
    target_total_val += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_val: {input_total_val}")
print(f"==>> target_total_val: {target_total_val}")
print(f"==>> input_total_val[unk_idx]: {input_total_val[unk_idx]}")
print(f"==>> target_total_val[unk_idx]: {target_total_val[unk_idx]}")

input_total_test = Counter([])
target_total_test = Counter([])
# total

num_batches_test = len(loaders.loader_test)

for step, batch in tqdm(
    enumerate(loaders.loader_test),
    total=num_batches_test,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_test += input_count
    target_total_test += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_test: {input_total_test}")
print(f"==>> target_total_test: {target_total_test}")
print(f"==>> input_total_test[unk_idx]: {input_total_test[unk_idx]}")
print(f"==>> target_total_test[unk_idx]: {target_total_test[unk_idx]}")

==>> unk_idx: 2


100%|██████████| 20031/20031 [01:45<00:00, 190.18it/s]

==>> input_total: Counter({0: 38628904, 1: 1281934, 3: 1231055, 4: 427959, 6: 313487, 15: 292084, 9: 272351, 11: 247959, 22: 227423, 12: 209259, 21: 172451, 35: 160825, 19: 143782, 32: 136435, 31: 126844, 25: 115514, 7: 109451, 42: 107337, 34: 105288, 30: 105192, 27: 102854, 18: 102613, 37: 99941, 24: 96661, 36: 94822, 41: 94217, 26: 92455, 56: 91259, 39: 86645, 53: 81602, 45: 80818, 47: 80376, 49: 74250, 50: 68885, 85: 66187, 44: 61816, 129: 59945, 52: 59210, 48: 58087, 59: 55220, 190: 52205, 88: 51717, 78: 50705, 79: 50356, 51: 49748, 74: 47617, 72: 46428, 70: 46107, 63: 45927, 57: 44254, 86: 43414, 58: 43001, 76: 41692, 80: 40317, 75: 37144, 105: 37027, 82: 33088, 121: 31609, 77: 31494, 66: 31338, 68: 31211, 208: 30686, 145: 30501, 125: 29147, 164: 29105, 99: 29080, 124: 29025, 95: 28116, 100: 27495, 98: 27376, 196: 27223, 600: 27209, 220: 26669, 123: 26205, 94: 25919, 61: 25905, 108: 25838, 156: 25168, 328: 25057, 127: 24623, 153: 24401, 132: 24160, 178: 24116, 119: 23869, 479: 237


100%|██████████| 4507/4507 [00:22<00:00, 202.02it/s]

==>> input_total_val: Counter({0: 8648372, 1: 288435, 3: 277030, 4: 96234, 6: 70050, 15: 65312, 9: 61727, 11: 55818, 22: 51026, 12: 47156, 21: 38417, 35: 35816, 19: 32310, 32: 30509, 31: 28310, 25: 25998, 7: 24491, 42: 24238, 30: 23887, 34: 23834, 27: 23292, 18: 23196, 37: 22267, 24: 21839, 36: 21332, 41: 20999, 26: 20933, 56: 20582, 39: 19484, 53: 18318, 47: 18241, 45: 18187, 49: 17046, 50: 15490, 85: 14824, 44: 13970, 129: 13634, 52: 13223, 48: 13134, 59: 12428, 190: 11757, 78: 11578, 88: 11400, 79: 11314, 51: 11144, 70: 10611, 74: 10590, 72: 10382, 63: 10291, 57: 10197, 86: 9870, 58: 9588, 76: 9385, 80: 9175, 75: 8354, 105: 8261, 82: 7517, 145: 7031, 121: 7022, 77: 6950, 66: 6864, 68: 6806, 208: 6794, 164: 6644, 125: 6618, 95: 6529, 99: 6525, 124: 6467, 100: 6203, 196: 6139, 600: 6082, 98: 6059, 220: 5885, 94: 5835, 123: 5808, 61: 5750, 108: 5739, 127: 5633, 156: 5596, 328: 5575, 178: 5397, 153: 5381, 479: 5362, 160: 5333, 163: 5238, 119: 5228, 327: 5220, 109: 5196, 132: 5137, 202: 


100%|██████████| 501/501 [00:02<00:00, 220.75it/s]


==>> input_total_test: Counter({0: 954386, 1: 32049, 3: 30865, 4: 10814, 6: 7593, 15: 7384, 9: 6733, 11: 6264, 22: 5635, 12: 5293, 21: 4359, 35: 4094, 19: 3595, 32: 3418, 31: 3169, 25: 2853, 42: 2802, 7: 2780, 34: 2675, 27: 2651, 30: 2640, 18: 2585, 24: 2522, 37: 2471, 41: 2388, 26: 2354, 36: 2349, 56: 2334, 39: 2160, 47: 2054, 53: 2033, 45: 2018, 49: 1812, 50: 1718, 85: 1622, 52: 1511, 44: 1501, 129: 1487, 48: 1443, 59: 1334, 88: 1317, 190: 1282, 78: 1262, 79: 1219, 74: 1216, 51: 1188, 72: 1162, 70: 1154, 86: 1123, 57: 1072, 63: 1071, 58: 1054, 76: 1025, 80: 995, 105: 921, 75: 890, 82: 832, 121: 818, 77: 809, 66: 788, 68: 761, 145: 754, 164: 746, 208: 735, 196: 709, 95: 707, 99: 696, 125: 695, 98: 683, 124: 683, 100: 681, 123: 679, 94: 650, 220: 649, 61: 642, 600: 635, 127: 631, 479: 625, 108: 625, 156: 620, 202: 608, 163: 603, 178: 598, 328: 594, 153: 591, 12027: 581, 253: 578, 132: 578, 109: 577, 160: 574, 181: 574, 119: 569, 172: 561, 251: 559, 162: 557, 327: 556, 182: 552, 234: 54

In [ ]:
# tokenizer_name = "LGAI-EXAONE/K-EXAONE-236B-A23B"

data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv"
max_token_length = 512
batch_size_train = 64
num_workers = 4
batch_size_val = 64
batch_size_test = 64
val_num_workers = 4
start_idx = 1
end_idx = 53
padding_idx = 0
unk_idx = 3
seed = 42

In [ ]:
class LoadersLGEXA236():
    def __init__(
            self,
            data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",
            max_token_length = 512,
            batch_size_train = 8,
            num_workers = 4,
            batch_size_val = 4,
            batch_size_test = 4,
            val_num_workers = 4,
            start_idx = 64100, 
            end_idx = 1, 
            padding_idx = 0, 
            unk_idx = 2,
            seed = 42,
    ):
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.padding_idx = padding_idx
        self.unk_idx = unk_idx

        # 1) 데이터셋 로드
        dataset = load_dataset("csv", data_files=data_path)['train']
        # AI hub 한국어-영어 번역(병렬) 말뭉치 데이터셋
        # kor, en, cat 3개의 칼럼으로 구성

        # 2) 카테고리 컬럼의 고유 클래스 찾아서 ClassLabel 객체 생성
        unique_classes = dataset.unique('cat')  
        # 'cat' 열에 문장 분류 
        # 0: 구어체
        # 1: 대화체
        # 2: 문어체_뉴스
        # 3: 문어체_한국문화
        # 4: 문어체_조례
        # 5: 문어체_지자체웹사이트

        print(f"==>> unique_classes: {unique_classes}")
        class_label = ClassLabel(names=unique_classes)

        # 3) 기존 컬럼 타입 변경 (캐스팅)
        dataset = dataset.cast_column('cat', class_label)

        # 4) stratify_by_column으로 분할
        train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='cat')
        valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='cat')

        dataset_dict = DatasetDict({
            'train': train_validtest['train'],
            'validation': valid_test['train'],
            'test': valid_test['test']
        })

        # print(dataset_dict)

        NUM_CPU = multiprocessing.cpu_count()
        # print(f"==>> NUM_CPU: {NUM_CPU}")

        # self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
        # @@@ 점포, 만료 등의 단어가 <unk>인 문제 발견 => 다른 토크나이저 사용?
        self.tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/K-EXAONE-236B-A23B")

        print(self.tokenizer.all_special_ids)
        print(self.tokenizer.all_special_tokens)

        print(f"==>> self.tokenizer.model_max_length: {self.tokenizer.model_max_length}")

        print(f"==>> len(self.tokenizer): {len(self.tokenizer)}")

        self.max_token_length = min(max_token_length, self.tokenizer.model_max_length)
        print(f"==>> self.max_token_length: {self.max_token_length}")

        # partial을 이용해 tokenizer, max_token_length 인자 고정
        cetf = partial(convert_examples_to_features, tokenizer=self.tokenizer, max_token_length=self.max_token_length)
        # @@@ convert_examples_to_features함수에서 examples가 첫번쨰 인자가 아니면 
        # @@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서 
        # @@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러 발생

        self.datasets = dataset_dict.map(
                                cetf,
                                # lambda examples: convert_examples_to_features(examples, tokenizer=self.tokenizer, max_token_length=self.max_token_length),
                                batched=True,
                                # 원 데이터 'en', 'kor', 'cat' 등의 칼럼을 지우려면
                                # remove_columns 인자 사용
                                # remove_columns=dataset_dict["train"].column_names,
                                num_proc=NUM_CPU)

        print(f"==>> self.datasets: {self.datasets}")

        self.train_set = self.datasets['train']
        self.val_set = self.datasets['validation']
        self.test_set = self.datasets['test']

        c_fn = partial(collate_fn, start_idx=self.start_idx, end_idx=self.end_idx, padding_idx=self.padding_idx, unk_idx=self.unk_idx)

        self.loader_train = DataLoader(self.train_set, batch_size=batch_size_train, collate_fn=c_fn, shuffle=True, num_workers=num_workers, pin_memory=True)
        # 학습시에만 shuffle=True
        self.loader_val = DataLoader(self.val_set, batch_size=batch_size_val, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)
        self.loader_test = DataLoader(self.test_set, batch_size=batch_size_test, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)

# def convert_examples_to_features(tokenizer, max_token_length, examples):
# @@@@@@@@@ examples가 첫번쨰 인자가 아니면 
# @@@@@@@@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서
# @@@@@@@@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러가 발생한다
def convert_examples_to_features(examples, tokenizer, max_token_length):
    model_inputs = tokenizer(examples['kor'],
                             text_target=examples['en'],
                             max_length=max_token_length, truncation=True)
    # 여기서 첫번째 인자와 두번째 인자의 순서를 바꾸면 한영 번역 대신 영한 번역용으로 인풋과 타겟이 토큰화된다
    return model_inputs


def collate_fn(batch, start_idx, end_idx, padding_idx, unk_idx):
    # print('Original:\n', batch)
    # print("".center(50, "-"))
    # batch는 [{'kor':..., 'en':..., 'cat':숫자, 'input_ids':[...], 'attention_mask':[1, ...], 'labels': [...]}, ...] 형태
    
    # keys = batch[0].keys()
    keys = ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels']
    # print(f"==>> keys: {keys}")
    # print("".center(50, "-"))
    
    # new_batch = {k:[] for k in keys}
    # new_batch['decoder_inputs'] = []
    
    # for b in batch:
    #     for k,v in b.items():
    #         if k == 'input_ids' or k == 'attention_mask':
    #             new_batch[k].append(torch.LongTensor(v))
    #         elif k == 'labels':
    #             new_batch[k].append(torch.LongTensor(v))
    #             new_batch['decoder_inputs'].append(torch.LongTensor([65001] + v[:-1]))
    #         else:
    #             new_batch[k].append(v)

    # key값별로 value 다 모으기
    new_batch = {k:[b[k] for b in batch] for k in keys}

    # list들 LongTensor로 변환
    new_batch['decoder_inputs'] = [torch.LongTensor([start_idx] + label[:-1]) for label in new_batch['labels']]
    # print(f"==>> new_batch['decoder_inputs']: {new_batch['decoder_inputs']}")
    new_batch['input_ids'] = [torch.LongTensor(inp) for inp in new_batch['input_ids']]
    new_batch['labels'] = [torch.LongTensor(label) for label in new_batch['labels']]
    new_batch['attention_mask'] = [torch.LongTensor(mask) for mask in new_batch['attention_mask']]

    new_batch['ntokens'] = sum([l.numel() for l in new_batch['labels']])

    # decoder input의 attention mask 생성
    new_batch['decoder_mask'] = [torch.ones_like(d_inp, dtype=torch.long) for d_inp in new_batch['decoder_inputs']]


    # 각 input과 target 텐서를 패딩
    padded_inputs = pad_sequence(new_batch['input_ids'], batch_first=True, padding_value=padding_idx)
    new_batch['input_ids'] = padded_inputs
    padded_decoder_inputs = pad_sequence(new_batch['decoder_inputs'], batch_first=True, padding_value=padding_idx)
    new_batch['decoder_inputs'] = padded_decoder_inputs
    padded_targets = pad_sequence(new_batch['labels'], batch_first=True, padding_value=padding_idx)
    new_batch['labels'] = padded_targets

    # attention 마스크는 패딩(padding_idx) 대신 False(0)을 입력
    padded_masks = pad_sequence(new_batch['attention_mask'], batch_first=True, padding_value=0)
    new_batch['attention_mask'] = padded_masks

    padded_d_masks = pad_sequence(new_batch['decoder_mask'], batch_first=True, padding_value=0)
    new_batch['decoder_mask'] = padded_d_masks
    
    return new_batch

In [ ]:
loaders = LoadersLGEXA236(
    data_path=data_path,
    max_token_length=max_token_length,
    batch_size_train=batch_size_train,
    num_workers=num_workers,
    batch_size_val=batch_size_val,
    batch_size_test=batch_size_test,
    val_num_workers=val_num_workers,
    start_idx=start_idx,
    end_idx=end_idx,
    padding_idx=padding_idx,
    unk_idx=unk_idx,
    seed=seed,
)

==>> unique_classes: [2, 4, 1, 0, 3, 5]
[1, 53, 3, 0]
['[BOS]', '<|endofturn|>', '[UNK]', '[PAD]']
==>> self.tokenizer.model_max_length: 1000000000000000019884624838656
==>> len(self.tokenizer): 153600
==>> self.max_token_length: 512
==>> self.datasets: DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1281934
    })
    validation: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 288435
    })
    test: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 32049
    })
})


In [ ]:
print(f"==>> unk_idx: {unk_idx}")

input_total = Counter([])
target_total = Counter([])
# total

num_batches_train = len(loaders.loader_train)

for step, batch in tqdm(
    enumerate(loaders.loader_train),
    total=num_batches_train,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total += input_count
    target_total += target_count

    # if step == 10:
    #     break

print(f"==>> input_total: {input_total}")
print(f"==>> target_total: {target_total}")
print(f"==>> input_total[unk_idx]: {input_total[unk_idx]}")
print(f"==>> target_total[unk_idx]: {target_total[unk_idx]}")


input_total_val = Counter([])
target_total_val = Counter([])
# total

num_batches_val = len(loaders.loader_val)

for step, batch in tqdm(
    enumerate(loaders.loader_val),
    total=num_batches_val,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_val += input_count
    target_total_val += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_val: {input_total_val}")
print(f"==>> target_total_val: {target_total_val}")
print(f"==>> input_total_val[unk_idx]: {input_total_val[unk_idx]}")
print(f"==>> target_total_val[unk_idx]: {target_total_val[unk_idx]}")


input_total_test = Counter([])
target_total_test = Counter([])
# total

num_batches_test = len(loaders.loader_test)

for step, batch in tqdm(
    enumerate(loaders.loader_test),
    total=num_batches_test,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_test += input_count
    target_total_test += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_test: {input_total_test}")
print(f"==>> target_total_test: {target_total_test}")
print(f"==>> input_total_test[unk_idx]: {input_total_test[unk_idx]}")
print(f"==>> target_total_test[unk_idx]: {target_total_test[unk_idx]}")

==>> unk_idx: 3


100%|██████████| 20031/20031 [02:07<00:00, 157.36it/s]


==>> input_total: Counter({0: 44664829, 375: 1252651, 582: 607045, 373: 430630, 378: 321639, 377: 277422, 634: 271210, 379: 241685, 905: 193872, 730: 193758, 696: 164609, 2373: 149445, 657: 136885, 380: 133980, 732: 126080, 4605: 125672, 1548: 114237, 382: 111850, 1075: 106630, 41728: 101319, 2376: 98249, 381: 96139, 1488: 90088, 715: 87677, 2030: 83403, 17005: 82821, 13456: 81352, 720: 80183, 383: 77830, 386: 76196, 385: 72259, 384: 70902, 858: 70308, 999: 69766, 369: 68060, 392: 66040, 643: 64209, 125365: 63976, 798: 62055, 1222: 61104, 79694: 55140, 48199: 49766, 14178: 49346, 853: 49174, 370: 47453, 2171: 44923, 2751: 42929, 2425: 42511, 18420: 40687, 125371: 40361, 10348: 39303, 16524: 38431, 1043: 36100, 1058: 35534, 125373: 34281, 698: 32750, 40379: 31476, 954: 30146, 816: 30115, 125366: 29572, 2517: 29454, 942: 28800, 1050: 28759, 1823: 28528, 650: 27024, 125372: 26941, 125399: 26577, 1965: 25779, 987: 25468, 764: 24649, 722: 24383, 800: 23934, 125383: 23904, 765: 22872, 902: 2

100%|██████████| 4507/4507 [00:25<00:00, 174.39it/s]


==>> input_total_val: Counter({0: 9971996, 375: 281878, 582: 136373, 373: 96778, 378: 72239, 377: 62027, 634: 60832, 379: 54812, 730: 43580, 905: 43526, 696: 37192, 2373: 34074, 657: 30958, 380: 30500, 732: 28406, 4605: 28089, 1548: 25645, 382: 25154, 1075: 23957, 41728: 22673, 2376: 22274, 381: 21308, 1488: 20217, 715: 19840, 17005: 18824, 2030: 18479, 13456: 18342, 720: 18080, 383: 17447, 386: 17169, 385: 16018, 858: 15925, 384: 15856, 999: 15802, 369: 15230, 392: 14787, 125365: 14485, 643: 14434, 1222: 14086, 798: 13843, 79694: 12345, 14178: 11209, 48199: 11034, 853: 10971, 370: 10704, 2171: 10184, 2425: 9737, 2751: 9617, 18420: 9230, 125371: 8964, 10348: 8872, 16524: 8662, 1043: 8032, 1058: 7892, 125373: 7730, 698: 7252, 40379: 7083, 954: 6734, 816: 6681, 125366: 6664, 2517: 6606, 1050: 6567, 942: 6527, 650: 6141, 125399: 6093, 1823: 6062, 125372: 5940, 1965: 5801, 987: 5740, 764: 5529, 125383: 5426, 722: 5419, 800: 5274, 765: 5133, 1081: 5080, 902: 4998, 125379: 4956, 712: 4942, 1

100%|██████████| 501/501 [00:02<00:00, 202.66it/s]


==>> input_total_test: Counter({0: 1096801, 375: 31389, 582: 15071, 373: 10871, 378: 7886, 377: 7003, 634: 6871, 379: 6057, 905: 4810, 730: 4730, 696: 4170, 2373: 3720, 657: 3473, 380: 3349, 4605: 3228, 732: 3122, 1548: 2838, 382: 2779, 1075: 2627, 2376: 2555, 41728: 2495, 381: 2318, 715: 2232, 1488: 2223, 17005: 2114, 2030: 2077, 13456: 2052, 720: 2034, 383: 2002, 386: 1869, 385: 1824, 858: 1793, 384: 1736, 999: 1713, 369: 1679, 643: 1627, 392: 1616, 125365: 1603, 798: 1564, 1222: 1511, 79694: 1399, 48199: 1259, 14178: 1245, 370: 1201, 853: 1187, 2171: 1099, 18420: 1082, 2425: 1042, 125371: 1025, 2751: 1020, 10348: 1018, 16524: 935, 125373: 889, 1043: 870, 1058: 832, 40379: 792, 698: 787, 2517: 766, 954: 765, 1050: 752, 942: 743, 125366: 724, 650: 710, 1823: 691, 816: 688, 125399: 670, 125372: 649, 1965: 649, 125383: 618, 722: 603, 987: 597, 764: 591, 22226: 575, 1877: 566, 1081: 565, 1818: 557, 1522: 554, 765: 548, 823: 545, 1126: 542, 712: 542, 902: 540, 800: 539, 125995: 528, 12543

In [ ]:
# tokenizer_name = "Translation-EnKo/exaone3-instrucTrans-v2-enko-7.8b"

data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv"
max_token_length = 512
batch_size_train = 64
num_workers = 4
batch_size_val = 64
batch_size_test = 64
val_num_workers = 4
start_idx = 1
end_idx = 361
padding_idx = 0
unk_idx = 3
seed = 42

In [ ]:
class LoadersLGEXAENKO():
    def __init__(
            self,
            data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",
            max_token_length = 512,
            batch_size_train = 8,
            num_workers = 4,
            batch_size_val = 4,
            batch_size_test = 4,
            val_num_workers = 4,
            start_idx = 64100, 
            end_idx = 1, 
            padding_idx = 0, 
            unk_idx = 2,
            seed = 42,
    ):
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.padding_idx = padding_idx
        self.unk_idx = unk_idx

        # 1) 데이터셋 로드
        dataset = load_dataset("csv", data_files=data_path)['train']
        # AI hub 한국어-영어 번역(병렬) 말뭉치 데이터셋
        # kor, en, cat 3개의 칼럼으로 구성

        # 2) 카테고리 컬럼의 고유 클래스 찾아서 ClassLabel 객체 생성
        unique_classes = dataset.unique('cat')  
        # 'cat' 열에 문장 분류 
        # 0: 구어체
        # 1: 대화체
        # 2: 문어체_뉴스
        # 3: 문어체_한국문화
        # 4: 문어체_조례
        # 5: 문어체_지자체웹사이트

        print(f"==>> unique_classes: {unique_classes}")
        class_label = ClassLabel(names=unique_classes)

        # 3) 기존 컬럼 타입 변경 (캐스팅)
        dataset = dataset.cast_column('cat', class_label)

        # 4) stratify_by_column으로 분할
        train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='cat')
        valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='cat')

        dataset_dict = DatasetDict({
            'train': train_validtest['train'],
            'validation': valid_test['train'],
            'test': valid_test['test']
        })

        # print(dataset_dict)

        NUM_CPU = multiprocessing.cpu_count()
        # print(f"==>> NUM_CPU: {NUM_CPU}")

        # self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
        # @@@ 점포, 만료 등의 단어가 <unk>인 문제 발견 => 다른 토크나이저 사용?
        self.tokenizer = AutoTokenizer.from_pretrained("Translation-EnKo/exaone3-instrucTrans-v2-enko-7.8b")

        special_tokens_dict = {'pad_token': '[PAD]'}
        self.tokenizer.add_special_tokens(special_tokens_dict)
        
        print(self.tokenizer.all_special_ids)
        print(self.tokenizer.all_special_tokens)

        print(f"==>> self.tokenizer.model_max_length: {self.tokenizer.model_max_length}")

        print(f"==>> len(self.tokenizer): {len(self.tokenizer)}")

        self.max_token_length = min(max_token_length, self.tokenizer.model_max_length)

        # partial을 이용해 tokenizer, max_token_length 인자 고정
        cetf = partial(convert_examples_to_features, tokenizer=self.tokenizer, max_token_length=self.max_token_length)
        # @@@ convert_examples_to_features함수에서 examples가 첫번쨰 인자가 아니면 
        # @@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서 
        # @@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러 발생

        self.datasets = dataset_dict.map(
                                cetf,
                                # lambda examples: convert_examples_to_features(examples, tokenizer=self.tokenizer, max_token_length=self.max_token_length),
                                batched=True,
                                # 원 데이터 'en', 'kor', 'cat' 등의 칼럼을 지우려면
                                # remove_columns 인자 사용
                                # remove_columns=dataset_dict["train"].column_names,
                                num_proc=NUM_CPU)

        print(f"==>> self.datasets: {self.datasets}")

        self.train_set = self.datasets['train']
        self.val_set = self.datasets['validation']
        self.test_set = self.datasets['test']

        c_fn = partial(collate_fn, start_idx=self.start_idx, end_idx=self.end_idx, padding_idx=self.padding_idx, unk_idx=self.unk_idx)

        self.loader_train = DataLoader(self.train_set, batch_size=batch_size_train, collate_fn=c_fn, shuffle=True, num_workers=num_workers, pin_memory=True)
        # 학습시에만 shuffle=True
        self.loader_val = DataLoader(self.val_set, batch_size=batch_size_val, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)
        self.loader_test = DataLoader(self.test_set, batch_size=batch_size_test, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)

# def convert_examples_to_features(tokenizer, max_token_length, examples):
# @@@@@@@@@ examples가 첫번쨰 인자가 아니면 
# @@@@@@@@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서
# @@@@@@@@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러가 발생한다
def convert_examples_to_features(examples, tokenizer, max_token_length):
    model_inputs = tokenizer(examples['kor'],
                             text_target=examples['en'],
                             max_length=max_token_length, truncation=True)
    # 여기서 첫번째 인자와 두번째 인자의 순서를 바꾸면 한영 번역 대신 영한 번역용으로 인풋과 타겟이 토큰화된다
    return model_inputs


def collate_fn(batch, start_idx, end_idx, padding_idx, unk_idx):
    # print('Original:\n', batch)
    # print("".center(50, "-"))
    # batch는 [{'kor':..., 'en':..., 'cat':숫자, 'input_ids':[...], 'attention_mask':[1, ...], 'labels': [...]}, ...] 형태
    
    # keys = batch[0].keys()
    keys = ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels']
    # print(f"==>> keys: {keys}")
    # print("".center(50, "-"))
    
    # new_batch = {k:[] for k in keys}
    # new_batch['decoder_inputs'] = []
    
    # for b in batch:
    #     for k,v in b.items():
    #         if k == 'input_ids' or k == 'attention_mask':
    #             new_batch[k].append(torch.LongTensor(v))
    #         elif k == 'labels':
    #             new_batch[k].append(torch.LongTensor(v))
    #             new_batch['decoder_inputs'].append(torch.LongTensor([65001] + v[:-1]))
    #         else:
    #             new_batch[k].append(v)

    # key값별로 value 다 모으기
    new_batch = {k:[b[k] for b in batch] for k in keys}

    # list들 LongTensor로 변환
    new_batch['decoder_inputs'] = [torch.LongTensor([start_idx] + label[:-1]) for label in new_batch['labels']]
    # print(f"==>> new_batch['decoder_inputs']: {new_batch['decoder_inputs']}")
    new_batch['input_ids'] = [torch.LongTensor(inp) for inp in new_batch['input_ids']]
    new_batch['labels'] = [torch.LongTensor(label) for label in new_batch['labels']]
    new_batch['attention_mask'] = [torch.LongTensor(mask) for mask in new_batch['attention_mask']]

    new_batch['ntokens'] = sum([l.numel() for l in new_batch['labels']])

    # decoder input의 attention mask 생성
    new_batch['decoder_mask'] = [torch.ones_like(d_inp, dtype=torch.long) for d_inp in new_batch['decoder_inputs']]


    # 각 input과 target 텐서를 패딩
    padded_inputs = pad_sequence(new_batch['input_ids'], batch_first=True, padding_value=padding_idx)
    new_batch['input_ids'] = padded_inputs
    padded_decoder_inputs = pad_sequence(new_batch['decoder_inputs'], batch_first=True, padding_value=padding_idx)
    new_batch['decoder_inputs'] = padded_decoder_inputs
    padded_targets = pad_sequence(new_batch['labels'], batch_first=True, padding_value=padding_idx)
    new_batch['labels'] = padded_targets

    # attention 마스크는 패딩(padding_idx) 대신 False(0)을 입력
    padded_masks = pad_sequence(new_batch['attention_mask'], batch_first=True, padding_value=0)
    new_batch['attention_mask'] = padded_masks

    padded_d_masks = pad_sequence(new_batch['decoder_mask'], batch_first=True, padding_value=0)
    new_batch['decoder_mask'] = padded_d_masks
    
    return new_batch

In [ ]:
loaders = LoadersLGEXAENKO(
    data_path=data_path,
    max_token_length=max_token_length,
    batch_size_train=batch_size_train,
    num_workers=num_workers,
    batch_size_val=batch_size_val,
    batch_size_test=batch_size_test,
    val_num_workers=val_num_workers,
    start_idx=start_idx,
    end_idx=end_idx,
    padding_idx=padding_idx,
    unk_idx=unk_idx,
    seed=seed,
)

==>> unique_classes: [2, 4, 1, 0, 3, 5]
[1, 361, 3, 0]
['[BOS]', '[|endofturn|]', '[UNK]', '[PAD]']
==>> self.tokenizer.model_max_length: 1000000000000000019884624838656
==>> len(self.tokenizer): 102400
==>> self.datasets: DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1281934
    })
    validation: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 288435
    })
    test: Dataset({
        features: ['kor', 'en', 'cat', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 32049
    })
})


In [ ]:
print(f"==>> unk_idx: {unk_idx}")

input_total = Counter([])
target_total = Counter([])
# total

num_batches_train = len(loaders.loader_train)

for step, batch in tqdm(
    enumerate(loaders.loader_train),
    total=num_batches_train,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total += input_count
    target_total += target_count

    # if step == 10:
    #     break

print(f"==>> input_total: {input_total}")
print(f"==>> target_total: {target_total}")
print(f"==>> input_total[unk_idx]: {input_total[unk_idx]}")
print(f"==>> target_total[unk_idx]: {target_total[unk_idx]}")

input_total_val = Counter([])
target_total_val = Counter([])
# total

num_batches_val = len(loaders.loader_val)

for step, batch in tqdm(
    enumerate(loaders.loader_val),
    total=num_batches_val,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_val += input_count
    target_total_val += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_val: {input_total_val}")
print(f"==>> target_total_val: {target_total_val}")
print(f"==>> input_total_val[unk_idx]: {input_total_val[unk_idx]}")
print(f"==>> target_total_val[unk_idx]: {target_total_val[unk_idx]}")


input_total_test = Counter([])
target_total_test = Counter([])
# total

num_batches_test = len(loaders.loader_test)

for step, batch in tqdm(
    enumerate(loaders.loader_test),
    total=num_batches_test,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_test += input_count
    target_total_test += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_test: {input_total_test}")
print(f"==>> target_total_test: {target_total_test}")
print(f"==>> input_total_test[unk_idx]: {input_total_test[unk_idx]}")
print(f"==>> target_total_test[unk_idx]: {target_total_test[unk_idx]}")

==>> unk_idx: 3


100%|██████████| 20031/20031 [02:01<00:00, 164.78it/s]

==>> input_total: Counter({0: 51439188, 375: 1253775, 696: 932057, 657: 901982, 634: 880598, 2373: 712170, 643: 631781, 730: 626493, 582: 604052, 905: 525739, 732: 512419, 4605: 509293, 373: 436753, 773: 425953, 1130: 331095, 1075: 322202, 378: 321767, 853: 299924, 377: 277422, 41728: 265790, 13456: 265105, 379: 244175, 1548: 211102, 924: 205154, 715: 200305, 2662: 197434, 1060: 176567, 720: 158742, 868: 155171, 1371: 145086, 798: 143460, 698: 139901, 1222: 135246, 380: 134139, 2030: 133787, 999: 127507, 858: 125226, 2870: 124025, 48199: 118789, 369: 115016, 370: 114409, 382: 111850, 13452: 105417, 40379: 105183, 2957: 104931, 18420: 104661, 1965: 103937, 16524: 99859, 2376: 98248, 712: 98247, 17005: 97047, 381: 96140, 50747: 94665, 722: 91333, 838: 91316, 1488: 90088, 14178: 88100, 9583: 85093, 2171: 82365, 1145: 79854, 721: 77973, 383: 77831, 386: 76196, 1043: 75422, 1877: 73660, 385: 72259, 384: 70902, 942: 70664, 691: 69683, 4264: 69681, 97300: 69562, 392: 66071, 1107: 65758, 2751:


100%|██████████| 4507/4507 [00:24<00:00, 184.62it/s]


==>> input_total_val: Counter({0: 11505133, 375: 282161, 696: 209656, 657: 202546, 634: 198235, 2373: 160972, 643: 142069, 730: 140801, 582: 135684, 905: 118611, 732: 115393, 4605: 114477, 373: 98158, 773: 95620, 1130: 74083, 378: 72266, 1075: 72007, 853: 67346, 377: 62027, 13456: 59774, 41728: 59489, 379: 55297, 1548: 47283, 924: 46322, 715: 44940, 2662: 44425, 1060: 39366, 720: 35616, 868: 34502, 1371: 32454, 798: 32189, 698: 31110, 1222: 30807, 380: 30532, 2030: 29962, 999: 28858, 858: 28284, 2870: 27415, 48199: 26661, 369: 25917, 370: 25816, 382: 25155, 40379: 23801, 18420: 23644, 13452: 23598, 1965: 23426, 2957: 23399, 16524: 22602, 2376: 22274, 712: 22191, 17005: 21938, 50747: 21317, 381: 21309, 722: 20325, 838: 20290, 1488: 20217, 14178: 19823, 9583: 19113, 2171: 18776, 1145: 17832, 721: 17682, 383: 17448, 386: 17169, 1043: 17020, 1877: 16356, 385: 16019, 384: 15857, 942: 15736, 4264: 15477, 691: 15463, 97300: 15315, 392: 14792, 1107: 14764, 2751: 14235, 3401: 13782, 1367: 13604

100%|██████████| 501/501 [00:02<00:00, 217.43it/s]

==>> input_total_test: Counter({0: 1267146, 375: 31426, 696: 23516, 657: 22589, 634: 22424, 2373: 17778, 643: 15780, 730: 15281, 582: 14985, 905: 12947, 732: 12901, 4605: 12628, 373: 11049, 773: 10772, 1130: 8525, 1075: 8041, 378: 7888, 853: 7499, 377: 7003, 41728: 6713, 13456: 6532, 379: 6103, 1548: 5298, 924: 5101, 715: 4993, 2662: 4935, 1060: 4519, 720: 4037, 868: 3949, 1371: 3697, 798: 3639, 698: 3536, 2030: 3357, 380: 3349, 1222: 3348, 999: 3178, 858: 3105, 2870: 3093, 48199: 2975, 369: 2867, 370: 2860, 382: 2779, 18420: 2734, 40379: 2648, 1965: 2605, 2957: 2574, 2376: 2555, 13452: 2552, 16524: 2515, 17005: 2493, 712: 2448, 50747: 2389, 838: 2364, 381: 2318, 722: 2302, 1488: 2223, 14178: 2184, 9583: 2123, 2171: 2070, 1145: 2036, 383: 2002, 386: 1869, 1877: 1832, 942: 1828, 721: 1826, 385: 1824, 4264: 1820, 1043: 1812, 384: 1736, 97300: 1730, 691: 1702, 392: 1618, 1107: 1587, 1367: 1489, 1522: 1485, 3401: 1482, 2751: 1477, 916: 1443, 650: 1430, 5623: 1391, 1216: 1387, 941: 1385, 10

# en_ko_12m 데이터셋

In [1]:
from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict

In [34]:
ds = load_from_disk("/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m")

In [35]:
ds

DatasetDict({
    train: Dataset({
        features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text'],
        num_rows: 11895711
    })
})

In [36]:
unique_classes = ds.unique('domain')  
print(unique_classes)

{'train': ['일상생활', '해외고객과의채팅', '해외영업', '경제', '기술과학', '기후', '세계', '정치', '교양', '다큐', '연예공연', '영화드라마', '예능오락', '인터뷰', '기타', '다큐교양', '01000', '02000', '03000', '10110', '10121', '10122', '10129', '10210', '10220', '10300', '10400', '10500', '10610', '10620', '10712', '10713', '10720', '10730', '10742', '10743', '10749', '10750', '10791', '10792', '10795', '10796', '10797', '10799', '10801', '10802', '11110', '11120', '11200', '12000', '13103', '13104', '13109', '13213', '13219', '13221', '13224', '13229', '13400', '13920', '13992', '13994', '13999', '14120', '14191', '14192', '14194', '14199', '14411', '14419', '14491', '14499', '15129', '15211', '15219', '15220', '16101', '16102', '16103', '16211', '16212', '16221', '16229', '16230', '16300', '17110', '17120', '17220', '17901', '17902', '17909', '18110', '18120', '18200', '19210', '19221', '19229', '20111', '20119', '20121', '20129', '20131', '20132', '20201', '20202', '20203', '20311', '20312', '20313', '20321', '20322', '20411', '20413'

In [ ]:
count = 0

def map_domain(raw_domain: str) -> str:
    if raw_domain is None:
        return "기타"

    d = raw_domain

    # 1) 명시적인 매핑
    if d in ["일상생활", "구어체_대화체", "해외고객과의채팅", "해외영업"]:
        return "일상/대화"

    if d in ["뉴스문어체", "지자체웹사이트 문어체", "가정통신문", "스포츠"]:
        return "뉴스/시사"

    if d in ["교양", "다큐", "다큐교양", "연예공연", "영화드라마", "예능오락", "인터뷰", "기타", # 방송콘텐츠 데이터셋에 기타 분류가 있음
             "문화문어체", 
             "문화", "문화·예술", "문화·교육", "문화재", "민속", "생활·민속",
             "문화유산", "역사", "역사/근현대", "역사/전통 시대",
             "구비 전승·언어·문학", "성씨·인물",
             "관광", "예술", "향토문화/음식", "종교", "지리",
             "정치·경제·사회", "정치∙경제∙사회", # 종류가 두가지?
             "정치·경제·산업/경제·산업|역사/근현대", "한국국제문화교류진흥원"]:
        return "문화/예술/역사"
    

    if d in ["경제", "세계", "정치", "기후", "기술과학", # 기술과학 분야 한-영 번역 병렬 말뭉치 데이터
             "글로벌동향정보", "연구평가정보", "위해식품정보", "법제도정보", # 식품 전문 분야 데이터
             "IT/기술", "ICT", "컴퓨터과학", "정보-통신",
             "공학", "전기", "전자", "재료", "재료과학",
             "수학", "물리학", "미생물학", "화학", "생명과학", "생물학 생화학",
             "환경과 생태학", "농학", "농림수산식품", "기계",
             "사회", "사회과학", "전문분야 문어체", "교육"]:
        return "과학/기술/학술자료"

    if d in ["의료/보건", "보건의료", "의학", "의약학", "의약학", "약리학 독성학"]:
        return "의학/보건"

    if d in ["법률", "대법원판례", "조례문어체", "교통"]:
        return "법률/행정"

    if d in ["금융/증시"]:
        return "금융/경제"

    # 2) 패턴 기반 매핑 (복잡한 문자열들)
    if d.startswith("문화·교육") or d.startswith("문화유산") or d.startswith("생활·민속"):
        return "문화/예술/역사"
    if d.startswith("역사/"):
        return "문화/예술/역사"
    if d.startswith("종교/"):
        return "문화/예술/역사"
    if d.startswith("지리/"):
        return "문화/예술/역사"
    if d.startswith("정치·경제·사회"):
        return "문화/예술/역사"
    if "의료" in d or "의학" in d or "보건" in d:
        return "의학/보건"
    
    # 특허
    if d.isdigit():
        return "특허"
    
    if d in ["H", "K"]:
        return "특허"

    # 3) 그 밖의 코드/기타 값
    if d in ["None"]:
        return "기타"

    # 조건에서 벗어난 경우의 수 카운트
    global count 
    count += 1
    print(d)
    # 기본값
    return "기타"


In [38]:
def map_domain_batch(batch):
    batch["super_domain"] = [map_domain(d) for d in batch["domain"]]
    return batch

In [39]:
ds = ds.map(map_domain_batch, batched=True)

Map:   0%|          | 0/11895711 [00:00<?, ? examples/s]

In [40]:
print(f"==>> count: {count}")

==>> count: 0


In [45]:
ds

DatasetDict({
    train: Dataset({
        features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text', 'super_domain'],
        num_rows: 11895711
    })
})

In [43]:
ds["train"][11000000:11000005]

{'domain': ['None', 'None', 'None', 'None', 'None'],
 'subdomain': ['None', 'None', 'None', 'None', 'None'],
 'style': ['문어체', '문어체', '문어체', '문어체', '문어체'],
 'target': ['ko', 'ko', 'ko', 'ko', 'ko'],
 'source': ['en', 'en', 'en', 'en', 'en'],
 'target_text': ['원래 더 많은 인원으로 데뷔할 생각이 있었다.',
  '웰리힐리 리조트는 대대로 스노보더에게 인기가 많은 곳이다.',
  '위트가 가미된 절제된 톤으로 현실적인 이야기를 풀어냈다.',
  '유니클로 감사제가 많은 이들의 이목을 집중시키고 있다.',
  '유독 여성 연예인들에게는 엄격한 외모 잣대가 주어진다.'],
 'source_text': ['I originally planned to debut with more people.',
  'Wellihilli Resort is popular with snowboarders for generations.',
  'Witness adds a realistic story with modest tones.',
  'The special events of UNIQLO have attracted much attention.',
  'Female entertainers are given strict standards of appearance.'],
 'super_domain': ['기타', '기타', '기타', '기타', '기타']}

In [44]:
ds.save_to_disk("en_ko_12m_with_super_domain")

Saving the dataset (0/11 shards):   0%|          | 0/11895711 [00:00<?, ? examples/s]

### feature 정리

In [1]:
from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset
import pandas as pd

In [2]:
ds = load_from_disk("/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_with_super_domain")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text', 'super_domain'],
        num_rows: 11895711
    })
})

In [8]:
print(f'==>> type(ds["train"]["super_domain"]): {type(ds["train"]["super_domain"])}')

==>> type(ds["train"]["super_domain"]): <class 'datasets.arrow_dataset.Column'>


In [5]:
ds_filtered  = ds.filter(lambda x: x["super_domain"] != "기타")
ds_filtered

DatasetDict({
    train: Dataset({
        features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text', 'super_domain'],
        num_rows: 10393939
    })
})

In [4]:
df = ds["train"].to_pandas()

In [11]:
df.head()

,domain,subdomain,style,target,source,target_text,source_text,super_domain
0,일상생활,구매,구어체,ko,en,>제가 이걸 보여드리겠습니다.,>Let me show you this.,일상/대화
1,일상생활,구매,구어체,ko,en,">지금도, 지금도 보면 다들 고개를 끄덕끄덕했잖아.","> Even now, everyone is nodding.",일상/대화
2,일상생활,구매,구어체,ko,en,>와우영미.,> WaWooYoungmi.,일상/대화
3,일상생활,구매,구어체,ko,en,>그래요.,> Okay.,일상/대화
4,일상생활,구매,구어체,ko,en,없어도 돼요.,I don't need it.,일상/대화


In [4]:
df["super_domain"].value_counts()

super_domain
과학/기술/학술자료    3822044
일상/대화         2719332
기타            1501772
문화/예술/역사      1346542
의학/보건          720037
법률/행정          626520
뉴스/시사          619465
특허             359999
금융/경제          180000
Name: count, dtype: int64

In [5]:
df_f = df[df["super_domain"] != "기타"]

In [6]:
df_f["super_domain"].value_counts()

super_domain
과학/기술/학술자료    3822044
일상/대화         2719332
문화/예술/역사      1346542
의학/보건          720037
법률/행정          626520
뉴스/시사          619465
특허             359999
금융/경제          180000
Name: count, dtype: int64

In [8]:
df_f.head()

,domain,subdomain,style,target,source,target_text,source_text,super_domain
0,일상생활,구매,구어체,ko,en,>제가 이걸 보여드리겠습니다.,>Let me show you this.,일상/대화
1,일상생활,구매,구어체,ko,en,">지금도, 지금도 보면 다들 고개를 끄덕끄덕했잖아.","> Even now, everyone is nodding.",일상/대화
2,일상생활,구매,구어체,ko,en,>와우영미.,> WaWooYoungmi.,일상/대화
3,일상생활,구매,구어체,ko,en,>그래요.,> Okay.,일상/대화
4,일상생활,구매,구어체,ko,en,없어도 돼요.,I don't need it.,일상/대화


In [9]:
df_f = df_f.rename(columns={'target_text': 'kor', 'source_text': 'en'})

In [8]:
df_f.head()

,domain,subdomain,style,target,source,kor,en,super_domain
0,일상생활,구매,구어체,ko,en,>제가 이걸 보여드리겠습니다.,>Let me show you this.,일상/대화
1,일상생활,구매,구어체,ko,en,">지금도, 지금도 보면 다들 고개를 끄덕끄덕했잖아.","> Even now, everyone is nodding.",일상/대화
2,일상생활,구매,구어체,ko,en,>와우영미.,> WaWooYoungmi.,일상/대화
3,일상생활,구매,구어체,ko,en,>그래요.,> Okay.,일상/대화
4,일상생활,구매,구어체,ko,en,없어도 돼요.,I don't need it.,일상/대화


In [31]:
dup_check = df.duplicated(subset="target_text")

In [32]:
dup_check.head()

0    False
1    False
2    False
3    False
4    False
dtype: bool

In [33]:
dup_check.value_counts()

False    10740854
True      1154857
Name: count, dtype: int64

In [34]:
dup_check[dup_check == True]

58          True
72          True
101         True
111         True
112         True
            ... 
11895690    True
11895692    True
11895698    True
11895701    True
11895703    True
Length: 1154857, dtype: bool

In [7]:
ds_filtered["train"]

Dataset({
    features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text', 'super_domain'],
    num_rows: 10393939
})

In [8]:
df_f = ds_filtered["train"].to_pandas()


: 

In [ ]:
df_f["super_domain"].value_counts()

NameError: name 'ds_filtered' is not defined

In [13]:
etc = df[df["super_domain"] == "기타"]
etc.head()

,domain,subdomain,style,target,source,target_text,source_text,super_domain
10293293,None,None,구어체,ko,en,'Bible Coloring'은 성경의 아름다운 이야기를 체험 할 수 있는 컬러링 ...,Bible Coloring' is a coloring application that...,기타
10293294,None,None,구어체,ko,en,씨티은행에서 일하세요?,Do you work at a City bank?,기타
10293295,None,None,구어체,ko,en,푸리토의 베스트셀러는 해외에서 입소문만으로 4차 완판을 기록하였다.,"PURITO's bestseller, which recorded 4th rough ...",기타
10293296,None,None,구어체,ko,en,11장에서는 예수님이 이번엔 나사로를 무덤에서 불러내어 죽은 자 가운데서 살리셨습니다.,In Chapter 11 Jesus called Lazarus from the to...,기타
10293297,None,None,구어체,ko,en,"6.5, 7, 8 사이즈가 몇 개나 더 재입고 될지 제게 알려주시면 감사하겠습니다.",I would feel grateful to know how many stocks ...,기타


In [14]:
etc["style"].value_counts()

style
문어체    1001772
구어체     400000
대화체     100000
Name: count, dtype: int64

In [15]:
lit = etc[etc["style"] == "문어체"]
lit.head()

,domain,subdomain,style,target,source,target_text,source_text,super_domain
10793293,None,None,문어체,ko,en,스키너가 말한 보상은 대부분 눈으로 볼 수 있는 현물이다.,Skinner's reward is mostly eye-watering.,기타
10793294,None,None,문어체,ko,en,심지어 어떤 문제가 발생할 건지도 어느 정도 예측이 가능하다.,Even some problems can be predicted.,기타
10793295,None,None,문어체,ko,en,오직 하나님만이 그 이유를 제대로 알 수 있을 겁니다.,Only God will exactly know why.,기타
10793296,None,None,문어체,ko,en,중국의 논쟁을 보며 간과해선 안 될 게 기업들의 고충이다.,Businesses should not overlook China's dispute.,기타
10793297,None,None,문어체,ko,en,박자가 느린 노래는 오랜 시간이 지나 뜨는 경우가 있다.,Slow-beating songs often float over time.,기타


In [27]:
lit.sample(10)

,domain,subdomain,style,target,source,target_text,source_text,super_domain
11270382,None,None,문어체,ko,en,가민에 따르면 지난해 4월 가민 한국지사를 설립한 이후 올해 상반기 매출 성장이 지...,Since establishing its Korean branch in April ...,기타
11476363,None,None,문어체,ko,en,강원 강릉의 한 펜션에서 일산화탄소 중독으로 의식을 잃어 병원에서 치료를 받고 있는...,Students who are being treated at hospitals fo...,기타
11371189,None,None,문어체,ko,en,양 대표는 “솔직히 중소기업에서 안전을 고려해 상대적으로 비싼 장비를 구입하는 것은...,"CEO Yang said, ""To be honest, it is not easy f...",기타
11388112,None,None,문어체,ko,en,"귀화한 라틀리프를 보유한 현대모비스는 외국인 선수 2명이 뛸 수 있는 2,3쿼터에 ...","Hyundai Mobis, which owns the naturalized Ratl...",기타
10991833,None,None,문어체,ko,en,노조는 “권력을 감시하던 언론인이 하루 아침에 권력 핵심부의 공직자로 자리를 옮겼다...,"""A journalist who had been monitoring power mo...",기타
11208222,None,None,문어체,ko,en,폭스콘은 중국·동남아와 비교해 인건비가 비싼 일본에서 가전을 생산하는 것은 수지에 ...,Foxconn decided that producing household appli...,기타
11166881,None,None,문어체,ko,en,김용진 기획재정부 차관이 31일 서울시 서초구 반포대로 서울지방조달청에서 열린 제7...,Vice-minister Kim Yong-jin of the Ministry of ...,기타
11218964,None,None,문어체,ko,en,삼화페인트 컬러디자인센터(이하 삼화페인트)가 지난달26일 서울 삼성동 코엑스에서 열...,The Samhwa Paints Color Design Center (Samhwa ...,기타
10950960,None,None,문어체,ko,en,유럽 최대 경제강국인 독일이 미국이 제공하는 안보에 안주하면서 러시아와 천연가스 사...,The purpose is that the biggest economic power...,기타
11759927,None,None,문어체,ko,en,이것을 옮기라고 하면 한 번 옮기려면 이미 400억 이상이 투자된 것이 손실이다.,"If you ask to move it, you will lose the inves...",기타


In [16]:
aihub = load_dataset("csv", data_files="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",)

In [17]:
aihub

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat'],
        num_rows: 1602418
    })
})

In [18]:
df_ai = aihub["train"].to_pandas()
df_ai.head()

,kor,en,cat
0,그 이후 그의 행방이 알려지지 않은 상황이다.,"Since then, his whereabouts are unknown.",2
1,그녀가 사라진 후 각종 의혹은 꼬리를 물었다.,Various doubts arose after she disappeared.,2
2,당연히 러시아인 승객이 제일 먼저 부름을 받았다.,"Naturally, Russian passengers were called first.",2
3,더 중요한 것은 고용이 대체적으로 안정됐다는 점이다.,"More importantly, employment is generally stable.",2
4,모세가 죽고 나서 이제 여호수아가 이스라엘을 이끌었습니다.,Joshua led Israel after Moses died.,2


In [20]:
df_ai[df_ai["cat"] == 0].head()

,kor,en,cat
400298,'Bible Coloring'은 성경의 아름다운 이야기를 체험 할 수 있는 컬러링 ...,Bible Coloring' is a coloring application that...,0
400299,씨티은행에서 일하세요?,Do you work at a City bank?,0
400300,푸리토의 베스트셀러는 해외에서 입소문만으로 4차 완판을 기록하였다.,"PURITO's bestseller, which recorded 4th rough ...",0
400301,11장에서는 예수님이 이번엔 나사로를 무덤에서 불러내어 죽은 자 가운데서 살리셨습니다.,In Chapter 11 Jesus called Lazarus from the to...,0
400302,"6.5, 7, 8 사이즈가 몇 개나 더 재입고 될지 제게 알려주시면 감사하겠습니다.",I would feel grateful to know how many stocks ...,0


In [21]:
df_ai[df_ai["cat"] == 1].head()

,kor,en,cat
300298,이번 신제품 출시에 대한 시장의 반응은 어떤가요?,How is the market's reaction to the newly rele...,1
300299,판매량이 지난번 제품보다 빠르게 늘고 있습니다.,The sales increase is faster than the previous...,1
300300,그렇다면 공장에 연락해서 주문량을 더 늘려야겠네요.,"Then, we'll have to call the manufacturer and ...",1
300301,"네, 제가 연락해서 주문량을 2배로 늘리겠습니다.","Sure, I'll make a call and double the volume o...",1
300302,지난 회의 마지막에 논의했던 안건을 다시 볼까요?,Shall we take a look at the issues we discusse...,1


In [22]:
df_ai[df_ai["cat"] == 2].head()

,kor,en,cat
0,그 이후 그의 행방이 알려지지 않은 상황이다.,"Since then, his whereabouts are unknown.",2
1,그녀가 사라진 후 각종 의혹은 꼬리를 물었다.,Various doubts arose after she disappeared.,2
2,당연히 러시아인 승객이 제일 먼저 부름을 받았다.,"Naturally, Russian passengers were called first.",2
3,더 중요한 것은 고용이 대체적으로 안정됐다는 점이다.,"More importantly, employment is generally stable.",2
4,모세가 죽고 나서 이제 여호수아가 이스라엘을 이끌었습니다.,Joshua led Israel after Moses died.,2


In [23]:
df_ai[df_ai["cat"] == 3].head()

,kor,en,cat
600298,강릉 기생 매화가 등장하는 판소리 열두마당의 하나인 「강릉매화전」은 판소리 특유의 ...,"<Gangneung Maehwajeon>, one of the twelve mada...",3
600299,"다양한 미술관련 전시회의 개최, 각종 교육프로그램과 새로운 미술 사업을 개발하고 운...",The purpose of the establishment was to hold v...,3
600300,간장은 가정에서 담구던 재래식 간장과 공장에서 양조된 개량식 간장으로 나뉜다.,There are two main types of soy sauce: home-br...,3
600301,그는 한국에 발을 내딛은 최조의 가톨릭 사제로 한국 역사에서 중요한 위치를 차지하고...,He occupies a significant position in Korean h...,3
600302,"덕사의 창건은 신라 말 고려 초라고 전해지나 문헌이 없어 상세히 알 길이 없고, 현...",It is unclear regarding the foundation of Deok...,3


In [24]:
df_ai[df_ai["cat"] == 4].head()

,kor,en,cat
200000,의원의 회의규칙 제47조제1항,Article 47(1) of the Members' Meeting Rules,4
200001,ⓛ비공개회의록은 원고로서 보관한다.,(1) The non-public meeting minutes shall be ke...,4
200002,o 보고지연 훈계 3일이상,– Delay in reporting results in discipline at ...,4
200003,건물면적은 33제곱미터(전용면적) 이상이어야 한다.,The building area shall be at least 33 m² (exc...,4
200004,이 경우 「행정대집행법」을 준용한다.,"In such cases, the Vicarious Administrative Ex...",4


In [25]:
df_ai[df_ai["cat"] == 5].head()

,kor,en,cat
901485,"""경기도가 말산업 육성을 위해 총예산 245,193천원으로 2013년 경기도 용인시...","""The Gyeonggi provincial government announced ...",5
901486,"""경기도가 주최하고 경기FTA활용지원센터와 코트라가 주관한 이번 시장개척단은 지난 ...","""Organized by Gyeonggi provincial government a...",5
901487,"""경기도가 주최하고 경기도비정규직지원센터가 주관한 이번 교육은 공공 부문이 직·간접...","""Organized by Gyeonggi provincial government a...",5
901488,"""경기도가 주최하고 사단법인 한국장애인복지시설협회 경기도협회(대표자:정권)에서 주관...","""Organized by Gyeonggi provincial government a...",5
901489,"""경기도가 주최하고 인구보건복지협회 경기지회, 아이낳기좋은세상 경기운동본부가 주관하...","""Hosted by Gyeonggi provincial government, org...",5


In [28]:
df_ai[df_ai["kor"] == "스키너가 말한 보상은 대부분 눈으로 볼 수 있는 현물이다."]

,kor,en,cat
1402407,스키너가 말한 보상은 대부분 눈으로 볼 수 있는 현물이다.,Skinner's reward is mostly eye-watering.,2


In [19]:
df_ai["cat"]

0          2
1          2
2          2
3          2
4          2
          ..
1602413    2
1602414    2
1602415    2
1602416    2
1602417    2
Name: cat, Length: 1602418, dtype: int64

In [ ]:
# cat_names = ["구어체", "대화체", "문어체_뉴스", "문어체_한국문화", "문어체_조례", "문어체_지자체웹사이트"]

# super_domain
# 과학/기술/학술자료    3822044
# 일상/대화         2719332
# 문화/예술/역사      1346542
# 의학/보건          720037
# 법률/행정          626520
# 뉴스/시사          619465
# 특허             359999
# 금융/경제          180000
# Name: count, dtype: int64

In [ ]:
# ds_ai = DatasetDict(
#     {
#         "train": Dataset.from_pandas(df_ai.reset_index(drop=True))
#     }
# )

In [ ]:
# ds_ai

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat'],
        num_rows: 1602418
    })
})

In [19]:
ds_ai = aihub

In [20]:
# cat_names = ["구어체", "대화체", "문어체_뉴스", "문어체_한국문화", "문어체_조례", "문어체_지자체웹사이트"]

def map_domain(raw_domain) -> str:
    if raw_domain == 0:
        return "일상/대화"
    
    if raw_domain == 1:
        return "일상/대화"
    
    if raw_domain == 2:
        return "뉴스/시사"
    
    if raw_domain == 3:
        return "문화/예술/역사"
    
    if raw_domain == 4:
        return "법률/행정"

    if raw_domain == 5:
        return "뉴스/시사"
    
    return "기타"

In [21]:
def map_domain_batch(batch):
    batch["super_domain"] = [map_domain(c) for c in batch["cat"]]
    return batch

In [22]:
ds_ai = ds_ai.map(map_domain_batch, batched=True)

Map:   0%|          | 0/1602418 [00:00<?, ? examples/s]

In [23]:
ds_ai

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'cat', 'super_domain'],
        num_rows: 1602418
    })
})

In [24]:
df_ai_new = ds_ai["train"].to_pandas()

In [25]:
df_ai_new.head()

,kor,en,cat,super_domain
0,그 이후 그의 행방이 알려지지 않은 상황이다.,"Since then, his whereabouts are unknown.",2,뉴스/시사
1,그녀가 사라진 후 각종 의혹은 꼬리를 물었다.,Various doubts arose after she disappeared.,2,뉴스/시사
2,당연히 러시아인 승객이 제일 먼저 부름을 받았다.,"Naturally, Russian passengers were called first.",2,뉴스/시사
3,더 중요한 것은 고용이 대체적으로 안정됐다는 점이다.,"More importantly, employment is generally stable.",2,뉴스/시사
4,모세가 죽고 나서 이제 여호수아가 이스라엘을 이끌었습니다.,Joshua led Israel after Moses died.,2,뉴스/시사


In [26]:
df_ai_new["super_domain"].value_counts()

super_domain
뉴스/시사       901474
일상/대화       500000
문화/예술/역사    100646
법률/행정       100298
Name: count, dtype: int64

In [27]:
df_ai_new_new = df_ai_new.drop('cat', axis=1)
df_ai_new_new.head()

,kor,en,super_domain
0,그 이후 그의 행방이 알려지지 않은 상황이다.,"Since then, his whereabouts are unknown.",뉴스/시사
1,그녀가 사라진 후 각종 의혹은 꼬리를 물었다.,Various doubts arose after she disappeared.,뉴스/시사
2,당연히 러시아인 승객이 제일 먼저 부름을 받았다.,"Naturally, Russian passengers were called first.",뉴스/시사
3,더 중요한 것은 고용이 대체적으로 안정됐다는 점이다.,"More importantly, employment is generally stable.",뉴스/시사
4,모세가 죽고 나서 이제 여호수아가 이스라엘을 이끌었습니다.,Joshua led Israel after Moses died.,뉴스/시사


In [28]:
df_ai_new_new.to_csv("data_aihub.csv", index=False)

In [14]:
df_f.head()

,domain,subdomain,style,target,source,kor,en,super_domain
0,일상생활,구매,구어체,ko,en,>제가 이걸 보여드리겠습니다.,>Let me show you this.,일상/대화
1,일상생활,구매,구어체,ko,en,">지금도, 지금도 보면 다들 고개를 끄덕끄덕했잖아.","> Even now, everyone is nodding.",일상/대화
2,일상생활,구매,구어체,ko,en,>와우영미.,> WaWooYoungmi.,일상/대화
3,일상생활,구매,구어체,ko,en,>그래요.,> Okay.,일상/대화
4,일상생활,구매,구어체,ko,en,없어도 돼요.,I don't need it.,일상/대화


In [10]:
df_f_new = df_f.drop(['domain', 'subdomain', 'style', 'target', 'source'], axis=1)
df_f_new.head()

,kor,en,super_domain
0,>제가 이걸 보여드리겠습니다.,>Let me show you this.,일상/대화
1,">지금도, 지금도 보면 다들 고개를 끄덕끄덕했잖아.","> Even now, everyone is nodding.",일상/대화
2,>와우영미.,> WaWooYoungmi.,일상/대화
3,>그래요.,> Okay.,일상/대화
4,없어도 돼요.,I don't need it.,일상/대화


In [11]:
df_f_new["kor"] = df_f_new["kor"].str.lstrip(">")
df_f_new["en"] = df_f_new["en"].str.lstrip(">")

In [12]:
df_f_new["kor"] = df_f_new["kor"].str.lstrip()
df_f_new["en"] = df_f_new["en"].str.lstrip()

In [13]:
df_f_new.head()

,kor,en,super_domain
0,제가 이걸 보여드리겠습니다.,Let me show you this.,일상/대화
1,"지금도, 지금도 보면 다들 고개를 끄덕끄덕했잖아.","Even now, everyone is nodding.",일상/대화
2,와우영미.,WaWooYoungmi.,일상/대화
3,그래요.,Okay.,일상/대화
4,없어도 돼요.,I don't need it.,일상/대화


In [14]:
df_f_new["super_domain"].value_counts()

super_domain
과학/기술/학술자료    3822044
일상/대화         2719332
문화/예술/역사      1346542
의학/보건          720037
법률/행정          626520
뉴스/시사          619465
특허             359999
금융/경제          180000
Name: count, dtype: int64

In [15]:
df_f_new.to_csv("data.csv", index=False)

In [ ]:
# ds_new = DatasetDict(
#     {
#         "train": Dataset.from_pandas(df_f_new.reset_index(drop=True))
#     }
# )

: 

In [35]:
# df_result = pd.merge(df_f_new, df_ai_new_new, on="super_domain")

In [ ]:
# ds_ai_new = DatasetDict(
#     {
#         "train": Dataset.from_pandas(df_ai_new_new.reset_index(drop=True))
#     }
# )

: 

### 데이터셋 합치기

In [31]:
from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets
import pandas as pd

In [29]:
data = load_dataset("csv", data_files="data.csv",)
data

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'super_domain'],
        num_rows: 10393939
    })
})

In [30]:
aihub = load_dataset("csv", data_files="data_aihub.csv",)
aihub

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'super_domain'],
        num_rows: 1602418
    })
})

In [32]:
data["train"] = concatenate_datasets([data["train"], aihub["train"]])

In [33]:
data

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'super_domain'],
        num_rows: 11996357
    })
})

In [34]:
data["train"] = data["train"].sort("super_domain")

In [35]:
data["train"] = data["train"].rename_column("super_domain", "domain")

In [36]:
data

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'domain'],
        num_rows: 11996357
    })
})

In [37]:
data["train"][0:10]

{'kor': ['특정 지역에서는 순위 변동이 크게 나타나고 있는 점에 주목해야 한다.',
  'Post-2020 체제에서는 매5년마다 이전 수준보다 개선된 온실가스 감축 목표를 유엔에 제출해야 하는 래칫 메커니즘이 적용되므로, 국내 최대 온실가스 배출 원인 발전 부문의 감축 부담도 크게 증가할 전망이다.',
  '회귀 모델은 모두 통계적으로 유의하였고, 분산 팽창 계수(VIF)도 모두 2 내외의 양호한 값을 보여 다중 공선성의 문제도 크지 않음을 확인하였다.',
  'OOF에 대한 연구는 그 중요성에도 불구하고 다른 개발 재원에 대한 연구에 비해 매우 부족한 실정이며, 다음의 측면에서 한계를 보이고 있다.',
  '이에 기대효용 이론에 기초하여 투자자들의 고차 적률 위험에 대한 선호를 반영할 수 있는 일반적 성과 지표를 개발할 필요성이 대두되었다.',
  '본 연구에서 통제 변수로 사용되는 나머지 변수들은 선행 연구들의 결과와 일치하게 모두 기업 설명회 개최 기업과 그렇지 않은 기업에 차이가 있는 것으로 나타났다.',
  '최근에는 이러한 문제에서 벗어나고자 국민 계정상 비법인 개인 기업의 영업 잉여를 자영업 소득으로 간주하고 이를 노동 소득과 자본 소득으로 적절히 분할하는 방법을 사용하고 있다.',
  '만약 2012년 한국 선박 금융 공사가 설립되어 운영되었다고 가정하면 해운 업계가 현재와 같은 위기 상황으로까지 내몰리지는 않았을 수도 있었을 것이다.',
  '자녀 양육을 지원하기 위한 다양한 공적이전소득에서 영유아 자녀에 대한 지원을 분리하여 산출할 필요가 있다.',
  '지금까지 시도된 참여형 정책 결정 과정들 대부분은 위로부터의 주어진 장에서, 초대받은 자들이 주어진 임무를 수행하는 것이다.'],
 'en': ['It should be noted that there are significant ranking fluctuations in certain regions.',
  "Under the Post-2020 system, the ratchet me

In [38]:
data.save_to_disk("en_ko_12m_p")

Saving the dataset (0/10 shards):   0%|          | 0/11996357 [00:00<?, ? examples/s]

In [1]:
from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets
import pandas as pd

In [2]:
ds = load_from_disk("/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m")


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['domain', 'subdomain', 'style', 'target', 'source', 'target_text', 'source_text'],
        num_rows: 11895711
    })
})

In [4]:
df = ds["train"].to_pandas()

In [5]:
df["style"].value_counts()

style
문어체     5923098
구어체     3119332
None    2393282
특허       359999
대화체      100000
Name: count, dtype: int64

In [6]:
df[df["style"] == "None"].sample(10)

,domain,subdomain,style,target,source,target_text,source_text
7263363,의료/보건,None,None,ko,en,사망자 중 60세 이상 노인층은 516명으로 49.8%에 달한다.,"Among the deaths, 516 elderly people aged 60 o..."
7372658,의료/보건,None,None,ko,en,"전염병 대응과 같은 긴급 상황에서는 일부 단계를 생략해 기간을 단축할 수 있지만, ...",In emergency situations such as a response to ...
6410470,IT/기술,None,None,ko,en,JW홀딩스는 자회사인 JW생명과학의 3체임버 종합영양수액제가 '현재 세계일류상품'에...,JW Holdings announced on the 22nd that its sub...
6476765,가정통신문,None,None,ko,en,"아울러, 유행하는 대부분의 감염병이 손을 통해 우리 몸으로 들어옵니다.","In addition, most of the prevalent infectious ..."
4267751,연예공연,연예가중계,None,ko,en,친구들은 제 광고를 보면 많이 놀려요.,My friends tease me a lot when they see my ads.
7412754,향토문화/음식,None,None,ko,en,"동쪽으로 이승골산과 짓당산, 남쪽으로 불무골, 북쪽으로 봉화산이 있으며, 서쪽으로 ...",It borders Iseunggolsan Mountain and Jidangsan...
5561257,생물학 생화학,None,None,ko,en,이를 위해 효과적인 모기 방역 기법을 도출하기 위해 모기 서식 환경을 분석하여 세종...,"To this end, in order to derive effective mosq..."
6201091,화학,화학,None,ko,en,뉴질랜드는 학교 단위 수준에서 교사들의 자율성과 전문성을 존중하고 학교별로 핵심 역...,New Zealand has a number of programs at the sc...
6585771,관광,None,None,ko,en,철근과 콘크리트의 열팽창 계수가 거의 같다는 것은 우리 인류에게 축복과 같은 일이었다.,It was a blessing for us humans to have almost...
6379244,IT/기술,None,None,ko,en,국토부는 신분증 미소지 승객의 불편을 해소하기 위해 정부가 발행하는 전자증명으로 신...,"The Ministry of Land, Infrastructure and Trans..."


In [6]:
df["domain"].value_counts()

domain
None                             1501772
해외영업                             1260367
일상생활                              899757
경제                                548114
해외고객과의채팅                          540221
                                  ...   
역사/근현대|문화·교육/언론·출판|문화유산/기록 유산          1
역사/근현대|종교/기독교|성씨·인물/근현대인물              1
역사/근현대|정치·경제·사회/경제 산업                  1
역사/근현대|인물/근현대 인물                       1
정치·경제·사회/경제·산업|문화·교육/문화·예술             1
Name: count, Length: 664, dtype: int64

In [7]:
df[df["domain"] == "해외고객과의채팅"].sample(10)

,domain,subdomain,style,target,source,target_text,source_text
1952364,해외고객과의채팅,도소매유통,구어체,ko,en,"하지만 비용이 50,000달러가 넘습니다.","But it would cost you over 50,000 dollars."
498574,해외고객과의채팅,도소매유통,구어체,ko,en,하지만 점점 플랫폼도 많아지고 경쟁이 치열합니다.,"However, there are more and more platforms and..."
1813138,해외고객과의채팅,"금융,보험",구어체,ko,en,시간이 충분하지 않았습니다.,We did not have enough time.
1917104,해외고객과의채팅,도소매유통,구어체,ko,en,"네, 물론입니다, 기다릴 수 있습니다.","Yes, sure, I can wait."
577687,해외고객과의채팅,도소매유통,구어체,ko,en,기존의 제품을 사용해도 되는지의 여부를 판단해서 준비하시면 되겠습니다.,You can decide whether you can use the existin...
2043666,해외고객과의채팅,정보통신,구어체,ko,en,대체불가 토큰에 대해 여쭤보고 싶습니다.,I just want to ask something about non-fungibl...
1940557,해외고객과의채팅,도소매유통,구어체,ko,en,제가 곧 이메일로 위탁 내역을 보내드리겠습니다.,I will send the consignment details through em...
456657,해외고객과의채팅,"기계장비,의료정밀",구어체,ko,en,계약서 초안에 몇 가지 불합리한 조항이 있는것 같아서요.,I think there are some unreasonable provisions...
708553,해외고객과의채팅,정보통신,구어체,ko,en,그런데 1안과 2안의 차이가 너무 커서 고민이 됩니다.,But I'm worried because it has a large differe...
1886022,해외고객과의채팅,도소매유통,구어체,ko,en,웹사이트에서 주문할 때의 제품 및 라벨의 스크린샷과 제 이메일 주소로 발송된 인보이...,I have a screenshot of the product and the lab...


In [8]:
df[df["domain"] == "해외영업"].sample(10)

,domain,subdomain,style,target,source,target_text,source_text
2619839,해외영업,"연구개발,과학기술",구어체,ko,en,우리가 많은 정보와 지식을 가지고 새로운 세계를 탐험할 수 있게 해줍니다.,"It lets us explore a new world, with much info..."
2483955,해외영업,도소매유통,구어체,ko,en,판매하는 제품인가요?,Are they included in the sale?
1104331,해외영업,도소매유통,구어체,ko,en,"집, 사무실, 이동중 어디서든 걱정없이 통화 가능합니다.","You can make calls without worries at home, in..."
1263158,해외영업,"연구개발,과학기술",구어체,ko,en,농도는 매우 신중하게 고려되어야 한다.,The concentration should be very carefully con...
802931,해외영업,도소매유통,구어체,ko,en,그러나 저는 공지 된 시간 내에 제품을 받지 못했습니다.,But I didn't receive the product within the an...
852699,해외영업,도소매유통,구어체,ko,en,식사량을 조절하고 체중을 관리하는데 효과적입니다.,It is effective in controlling the amount of m...
779610,해외영업,도소매유통,구어체,ko,en,그러나 저희는 아직 귀하로부터 입금을 받지 못했습니다.,But we haven't received a deposit from you yet.
1269221,해외영업,"연구개발,과학기술",구어체,ko,en,"Mr. Willson 선생님, 저희 쪽에서 생산하는 탄산 칼슘의 재료를 그쪽에서 받...","Mr. Wilson, I'd like to receive the ingredient..."
2239717,해외영업,도소매유통,구어체,ko,en,팬데믹 기간 중 당사는 해외 수출을 시작했습니다.,"During the pandemic, we started exporting our ..."
886539,해외영업,도소매유통,구어체,ko,en,저희는 300개에 대한 금액을 모두 지불했습니다.,We paid for all 300 pieces.


In [9]:
df[df["domain"] == "일상생활"].sample(10)

,domain,subdomain,style,target,source,target_text,source_text
166737,일상생활,여행,구어체,ko,en,>준비 시작하면 시작돼요.,>It starts when we get ready.
1441689,일상생활,여행,구어체,ko,en,소백산은 지방의 동쪽에 위치하고 있어요.,Mount Sobaek is located in the east of the pro...
237779,일상생활,예약,구어체,ko,en,>그런데 그렇게 해야 들어가.,>But you have to do it like that to go in.
1581256,일상생활,예약,구어체,ko,en,"프라이드 치킨, 프라이드 프라이, 버거와 같은 음식들이요.","Such as fried chicken, fried fries, and burgers."
801,일상생활,구매,구어체,ko,en,">야, 양세형 잘하네 양세형.",">Hey, Yang Se-hyeong is good. >Hey, Yang Se-hy..."
1551921,일상생활,여행,구어체,ko,en,그럼 먼저 좋은 점을 알려주세요.,Tell me the good ones first then.
1636865,일상생활,예약,구어체,ko,en,"AAA1, 추모식에서 아빠 추모연설을 해줘.","AAA1, I want you to perform the eulogy for Dad..."
236095,일상생활,예약,구어체,ko,en,그리고 만약에 부위에 부딪히면 그 자리 서있어야.,"And if you hit a part, you have to stand there."
281546,일상생활,음식,구어체,ko,en,> 아 그럼요 송지효도 지금 저한테 빠져 있잖아요.,"> Oh, of course. Song Jihyo is also into me ri..."
441803,일상생활,음식,구어체,ko,en,>어유~,>Uh~


In [10]:
df[df["domain"] == "구어체_대화체"].sample(10)

,domain,subdomain,style,target,source,target_text,source_text
5020911,구어체_대화체,None,구어체,ko,en,연중무휴 아니었어? 갑자기 왜 쉬는 건지 모르겠네.,Isn't it open throughout the year? What happened?
5020521,구어체_대화체,None,구어체,ko,en,그렇군요. 4주 동안 머물 곳은 정하셨나요? 확인할 방법이 있을까요?,I see. Have you confirmed where to stay for 4 ...
5025310,구어체_대화체,None,구어체,ko,en,이제 본다는 생각과 그냥 좋아서 이러고 있어.,I'm just doing like this with the thought that...
5011237,구어체_대화체,None,구어체,ko,en,"나는 하정우가 먹는 방식, 입는 방식이 좋아.",I like the way Ha Jung Woo eats and wears.
5015997,구어체_대화체,None,구어체,ko,en,그는 30분 전에 기차를 타러 나갔습니다.,He took off about half an hour ago to catch th...
5018325,구어체_대화체,None,구어체,ko,en,오백 미터 정도 걸어가면 돼요.,You just need to walk about 500 meters.
5020701,구어체_대화체,None,구어체,ko,en,혹시 위치를 잘못 말한 거 아닐까요? 확인 전화해 보세요.,Maybe you misdirected it. Give a call to check...
5029166,구어체_대화체,None,구어체,ko,en,제 생각에 이 배송 상자는 멀쩡한데 제품만 손상된 것으로 보아 배송 중 생긴 문제는...,Judging by the fact that the product was damag...
5024096,구어체_대화체,None,구어체,ko,en,"이 치즈로 수없이 많은 요리를 할 수 있지만, 튀김과 조림 요리가 가장 선호되죠.",You can cook countless dishes with this cheese...
5011250,구어체_대화체,None,구어체,ko,en,"나의 롤모델은 나의 언니, AAA입니다.","My role model is my older sister, AAA."


In [ ]:
# TODO: cat 칼럼 추가하기
# TODO: 중복 지우기

In [1]:
from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets
import pandas as pd

In [2]:
ds = load_from_disk("/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_p")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'domain'],
        num_rows: 11996357
    })
})

In [4]:
df = ds["train"].to_pandas()

In [5]:
df["domain"].value_counts()

domain
과학/기술/학술자료    3822044
일상/대화         3219332
뉴스/시사         1520939
문화/예술/역사      1447188
법률/행정          726818
의학/보건          720037
특허             359999
금융/경제          180000
Name: count, dtype: int64

In [8]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "과학/기술/학술자료"].sample(10)

,kor,en,domain
2464871,"어떤 실시예에 따르면, 상기 시선 추적 모듈(391)은 시선추적을 위한 마이크로 카...","According to an embodiment, the eye tracking m...",과학/기술/학술자료
2393326,상기 명령어 생성부는 상기 요청동작 판단부의 판단에 따른 사용자의 요청동작에 대응하...,The command generation unit may generate a com...,과학/기술/학술자료
2374490,다른 부재중 전화의 발신측은 전화 번호가 1850으로 시작하는 휴대폰 사용자이다.,The calling party of the other missed call is ...,과학/기술/학술자료
1191170,미 국무부는 한국 대표단과 면담을 통해 공산주의의 위협과 한국 승인에 있어서 미국의...,Through an interview with the Korean delegatio...,과학/기술/학술자료
1542896,"수학 교과가 생각하는 방법을 익히는 과목임을 고려한다면, 다양한 분류 가능성이 있음...",Considering that mathematics is a subject in w...,과학/기술/학술자료
3163410,생성된 바이오 디젤은 국제 기준에 못 미치게 되면 디젤의 성능을 저하시키고 사용의 ...,If the produced biodiesel does not meet the in...,과학/기술/학술자료
205606,호주 최대 항공사인 콴타스의 여객기가 하루 동안 세 대나 비상 착륙하는 사태가 발생했다.,"Australia's largest flight, Qantas, made a cri...",과학/기술/학술자료
765746,"통일신보는 ""위성 발사 역사는 겨울철에 위성을 쏴올려 성공한 예가 매우 낮다는 것을...","The Unification Shinbo reported, ""The history ...",과학/기술/학술자료
2364034,수신된 디지털 컴포넌트는 데이터 처리 시스템(102)의 탐색 컴포넌트(118)에 의...,The received digital component may be rendered...,과학/기술/학술자료
861608,맨체스터에서 전날 밤 공연장 테러로 적어도 22명이 숨진 일로 파운드화 가치가 하락...,At least 22 people were killed in the previous...,과학/기술/학술자료


In [9]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "일상/대화"].sample(10)

,kor,en,domain
10514542,처음에 투자로 구입했어요.,They were initially bought as investments.,일상/대화
8879564,스마트 온도/습도 자동 조절 기능이 있습니다.,It has a smart temperature/humidity automatic ...,일상/대화
11037338,이것은 이러한 로봇이 활동할 때 작업자의 감독이 거의 또는 전혀 없을 수 있음을 의...,This means that there could be little to no su...,일상/대화
11462590,"관세 지급 거부로 물건이 한국으로 반송되면 초기 출고 시 운임, 반송운임, 관세를 ...",If the product is returned to Korea due to rej...,일상/대화
8843209,어제?,Yesterday?,일상/대화
10205011,"아니, 불행히도 무제한 타코는 팔지 않아.","No, unfortunately, they don't serve unlimited ...",일상/대화
8873296,"환자가 끊이지 않고 있고, 특히 지금처럼 전염병이 도는 시기에는 특히 면역 검사 수...","The number of patients is constant, and the de...",일상/대화
10136270,"비건 다이어트를 할 수도 있고, 운동을 시작할 수도 있어요.","You can go on a vegan diet, or maybe start exe...",일상/대화
10650610,피부 톤이 고르지 않은 사람들에게 특히 효과적입니다.,It is especially useful for people with uneven...,일상/대화
10510359,또한 귀하의 비즈니스가 재정적 어려움을 겪을 때 대출을 제공합니다.,We also provide loans when your business exper...,일상/대화


In [10]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "뉴스/시사"].sample(10)

,kor,en,domain
4406395,체조에 대한 관심 확대 및 우수 선수 발굴·육성을 위한 제12회 서울특별시교육감배 ...,The 12th Seoul Metropolitan Office of Educatio...,뉴스/시사
5066818,민간 기업에 특정인의 채용이나 보직변경을 요구하거나 공공이 개최하는 축제나 박람회 ...,Civil servants will be banned from soliciting ...,뉴스/시사
5323215,예상에 크게 못 미친 애플의 실적에 세계 금융시장이 요동쳤다.,The global financial market fluctuated by Appl...,뉴스/시사
4388899,'학교생활기록 작성 및 관리 지침'을 근거로 작성한 『2019학년도 학교생활기록부 ...,We inform you of changes to the rules related ...,뉴스/시사
5185296,다음달 중 시가 안을 제출하면 국토부는 검토에 착수하게 되며 이르면 올해 안에 승인...,If the city submits a market price within next...,뉴스/시사
4215816,경기 종료 후 신태용 감독이 이용과 기뻐하고 있다.,"After the game ends, Shin Tae-yong rejoices wi...",뉴스/시사
5413659,지난 1월 소망교회에서 은퇴한 김지철(70) 목사가 새로이 ‘미래목회와 말씀연구원(...,"That's why Rev. Kim Ji-cheol, 70, who retired ...",뉴스/시사
5349867,생선 등 수분이 있는 제품을 담는 합성수지 제품과 채소 등의 제품을 담는 속 비닐은...,Synthetic resin products containing moisture p...,뉴스/시사
4577640,양키스를 비롯해 빅마켓 구단들의 경쟁이 붙으며 계약 규모가 커졌다.,As Yankees and other big clubs get into the co...,뉴스/시사
4563020,전반 34분엔 앤드류 로버트슨의 패스를 받은 베이날둠이 페널티지역 왼쪽에서 달려들며...,"In the 34th minute of the first half, Beinaldu...",뉴스/시사


In [11]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "문화/예술/역사"].sample(10)

,kor,en,domain
6727300,사회적 이슈에 대해 대중이 저마다 가진 다원적 의견들을 키치를 통해 소통할 수 있도...,This is because it allows the public to commun...,문화/예술/역사
6165038,서울 암사동유적의 유네스코(UNESCO) 세계문화유산 등재를 염원하며 화려한 막을 ...,"From the 11th of last month, when the festival...",문화/예술/역사
5784331,그리고 그 다음으로 비교할 제품은 이 두 가지 통심입니다.,And the next product to compare is these two m...,문화/예술/역사
6380516,"시도 19호선이 가송리의 중부를 동서 방향으로 지나고 있고, 기타 도로들이 가송리의...",City Road 19 runs east-west of the central par...,문화/예술/역사
6861041,"유리병으로 된 박카스는 다른 유리병과 부딪히는 과정에서 쉽게 깨지고, 종이로 된 라...","Bacchus, which was made up of a glass bottle, ...",문화/예술/역사
5767655,이제 같이 풀어야 하는 문제야.,Now it's a question we have to solve together.,문화/예술/역사
6113138,"이 밖에도 외국인들이 ‘코리안 범프’라고 하는 ‘어깨빵’, 폰딧불(스마트폰+반딧불이...","Newly coined words such as “Korean bump”, “sho...",문화/예술/역사
5841776,지금은 제가 메이크업까지 완성을 하고 왔습니다.,Now I have finished my makeup as well.,문화/예술/역사
6035241,그들은 이미 중력에 의해 방향 감각을 상실했다고 느낍니다.,They already feel disoriented by the force of ...,문화/예술/역사
6307077,그늘막은 햇볕을 가려주기도 하지만 때로는 어둠을 드리우는 위험한 상황을 만들기도 한다.,"The shade can cover the sun, but sometimes it ...",문화/예술/역사


In [12]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "법률/행정"].sample(10)

,kor,en,domain
6992409,구청장은 평생교육 운영비 등을 지원받은 기관 및 단체에 대하여 그 운영비 등을 목적...,"The head of the Gu may confirm, instruct, and ...",법률/행정
7648610,"운영위원회에는 간사 1명을 두되, 간사는 담당 교육연구사로 하고 위원장의 명을 받아...",The Steering Committee shall have one executiv...,법률/행정
7102099,"피고인은 2009. 3. 일자불상경 2항과 같은 장소에서, 같은 방법으로 피해자의 ...","On March, 2009, the Defendant forced an indece...",법률/행정
7379035,"단 그 제한은 자유와 권리의 본질적인 내용을 훼손하여서는 아니되며 언론, 출판에 대...","However, the restriction shall not undermine t...",법률/행정
7158751,"소송 지휘에 관한 결정·명령이나 집행정지 결정, 비송 사건에 관한 결정에는 기판력이...",There is no judgment in decisions or orders co...,법률/행정
7124449,불법행위 당시 두 가지 이상의 수입원에 해당하는 영업활동에 종사하고 있던 피해자의 ...,Where the amount of lost income of each busine...,법률/행정
7675025,주민자치회 위원을 선정함에 있어 투명하고 공정한 추첨 방법 결정과 과정 관리를 위하...,In selecting members of the Residents' Self-Go...,법률/행정
7441761,"그 결과, 다양한 기업지배구조 평가점수(주주 권리보호, 이사회 특성, 공시(IR 실...","As a result, various corporate governance eval...",법률/행정
7441837,"다만, 좀 더 부언하다면 상대적으로 노령계층과 취약계층이 많은 군지역의 경우 국고보...","If I were to say more, the military area, whic...",법률/행정
7366808,판례법 국가가 아니며 권력구조도 다른 한국에서 사용하기에는 적당하지 않은 면이 있는...,It is an expression inappropriate to be used i...,법률/행정


In [13]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "의학/보건"].sample(10)

,kor,en,domain
8311374,멕시코 추정치와 비교하여 시간 경과에 따른 변화를 평가하면 미국에서 사례 수와 발병...,"Compared to Mexican estimates, assessing chang...",의학/보건
8187872,하이드로 스캐폴드는 불수용성 겔 형성을 위해 가교를 하도록 자극하는 액체 폴리머이다.,Hydrogel scaffolds are liquid polymers that st...,의학/보건
7734268,"이들은 28개의 보명지주를 돕거나 강화하는 한약재를 비교 분석하여, 소음 체질 관련...",They compared and analyzed 28 herbal medicines...,의학/보건
7849381,뿌리 부분인 털망울이 위축되고 밑 부분이 탈색된 느낌표 모양의 모발도 흔히 관찰된다.,Exclamation mark-shaped hair with shrunk root ...,의학/보건
7900934,"입국할 때 발열 및 호흡기증상이 있다면 검역관에게 신고하여 역학조사에 협조하고, 메...",If you have any fever or respiratory symptoms ...,의학/보건
8078676,안전 사회를 위해 재난과 안전에 대한 법 체계가 중요하다.,The legal system for disaster and safety is im...,의학/보건
7747113,본 연구를 통해 선정된 11편의 문헌에서 7개의 허약 선별 평가 도구가 도출되었다.,Seven frailty screening evaluation tools were ...,의학/보건
8025708,이들에 대한 적절한 영양 공급차원의 경제적 지원제도 마련이 시급해 보인다.,There seems to be an urgent need to prepare an...,의학/보건
7798957,지난 29일 국내에서 코로나19 검사를 받고 있는 사람은 8634명이었다.,"On the 29th, 8,634 people were being tested fo...",의학/보건
8324176,악성 및 양성 상태를 별도로 진단하는 것은 임상 실습에서 빈번한 도전이다.,"Thus, separately diagnosing malignant and beni...",의학/보건


In [14]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "특허"].sample(10)

,kor,en,domain
11792601,"터보 차저 과열 방지 장치 및 상기 장치의 제조 방법본 발명에 따라, 미네랄 절연 ...",Turbocharger overheating protection device and...,특허
11670799,트로폴론 유도체레티노이드 작용을 가지며 의약의 유효성분으로서 유용한 화학식 I로 표...,TROPOLONE DERIVATIVETropolone derivatives repr...,특허
11983908,보안 메시지 제공 방법 및 시스템보안 메시지 제공 방법 및 시스템을 제공한다.본 발...,METHOD AND SYSTEM FOR PROVIDING SECURE MESSAGE...,특허
11690430,경화성 조성물(A) 성분 g당 10 내지 40 mg의 KOH인 산가를 갖고 자유 카...,Curable compositionCurable compositions which ...,특허
11923059,디스크 브레이크디자인판의 진동을 감소시킬 수 있는 디스크 브레이크를 제공한다. 캘리...,DISC BRAKETo provide a disc brake capable of r...,특허
11900846,양말연질 발포 시트를 이용한 양말에서의 소재의 강도를 약하게 하지 않고 통기성을 개...,SOCKSTo provide socks which uses a sot foamed ...,특허
11652920,합성 섬유용 처리제고연신 배율 고속 제사 조건에서도 조업성 자주(잘) 보풀의 적은 ...,TREATMENT AGENT FOR SYNTHETIC FIBERTo provide ...,특허
11758315,"코발트기 합금 제조물본 발명은, 석출 강화 Ni기 합금재와 동등 이상의 기계적 특성...",COBALT-BASED ALLOY PRODUCTThere is provided a ...,특허
11804822,자산 추적기자산 추적기 디바이스는 자산 추적기 디바이스의 배터리에 전력을 공급하기 ...,ASSET TRACKERAn asset tracker device includes ...,특허
11705525,공간 등 탈취 살균 장치공간 등 내를 효율적으로 탈취하거나 살균하는 공간 등 탈취 ...,APPARATUS FOR DEODORIZING AND STERILIZING INSI...,특허


In [15]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
df[df["domain"] == "금융/경제"].sample(10)

,kor,en,domain
3836256,주요 국제기구들은 중국의 성장세 하락이 내년에도 이어질 것으로 전망하고 있다.,Major international organizations predict that...,금융/경제
3900902,토스와 IMM PE 이외에도 국내 PG 업체와 글로벌 결제 서비스 업체 등이 도전장...,"In addition to Toss and IMM PE, domestic PG co...",금융/경제
3862367,손 회장 측이 법적 소송으로 가더라도 연임이 확정되는 것은 아니라는 지적도 나온다.,Some point out that even if Chairman Sohn goes...,금융/경제
3840685,시장에선 한은의 '10월 금리 인하' 전망이 대세론으로 굳어지는 분위기다.,"In the market, the outlook for the Bank of Kor...",금융/경제
3865618,이베이코리아가 운영하는 G9는 유료멤버십 '스마일클럽' 회원 전용관을 열었다고 14...,"G9, operated by eBay Korea, announced on the 1...",금융/경제
3909274,월별 주행거리는 보험 가입 시 캐롯손해보험이 제공하는 운행 데이터 측정 장치 '캐롯...,"Monthly mileage is measured by installing a ""C...",금융/경제
3869047,"아울러 공정위가 아시아나항공 기내식 공급과 관련, 박 전 회장과 전·현직 경영인을 ...","Moreover, in relation to the supply of Asiana ...",금융/경제
3893196,"장원귀 번개장터 대표는 ""번개장터에 차이를 도입함으로써 수수료 면제 혜택과 결제시간...","Jang Won-gwi, CEO of Bungaejangter, said, ""By ...",금융/경제
3946004,"한화생명보험, 한화손해보험, 교보라이프플래닛, DB손해보험, 메리츠화재 등 국내 대...",Eleven products have been prepared through par...,금융/경제
3859344,일반 사채보다 리스크가 높은 신종자본증권은 금리차가 더 벌어졌다.,"The interest gap of new capital securities, wh...",금융/경제


### 한 분야 안에 문어체, 구어체가 섞인 부분들도 보이고, 다른 분야인데도 비슷한 문장들이 보임 ==> 단순히 문어체, 구어체 2분류로?
- #### == 문어체, 구어체, 혼재 3분류
    - #### 문어체: 과학/기술/학술자료 뉴스/시사 법률/행정 의학/보건 특허 금융/경제
    - #### 구어체: 일상/대화
        - #### 일상/대화에 포함된 "해외고객과의채팅", "해외영업"의 경우 비교적 문체가 딱딱하지만 그래도 일단 구어체
    - #### 혼재: 문화/예술/역사

In [1]:
from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets
import pandas as pd

In [2]:
ds = load_from_disk("/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_p")

In [3]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
def map_domain(d: str) -> str:
    if d == "일상/대화":
        return "구어체"
    if d == "문화/예술/역사":
        return "혼재"
    
    return "문어체"
    

In [4]:
def map_domain_batch(batch):
    batch["super_domain"] = [map_domain(d) for d in batch["domain"]]
    return batch

In [5]:
ds = ds.map(map_domain_batch, batched=True)

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'domain', 'super_domain'],
        num_rows: 11996357
    })
})

In [6]:
df = ds["train"].to_pandas()
df.head()

,kor,en,domain,super_domain
0,특정 지역에서는 순위 변동이 크게 나타나고 있는 점에 주목해야 한다.,It should be noted that there are significant ...,과학/기술/학술자료,문어체
1,Post-2020 체제에서는 매5년마다 이전 수준보다 개선된 온실가스 감축 목표를 ...,"Under the Post-2020 system, the ratchet mechan...",과학/기술/학술자료,문어체
2,"회귀 모델은 모두 통계적으로 유의하였고, 분산 팽창 계수(VIF)도 모두 2 내외의...",All regression models were statistically signi...,과학/기술/학술자료,문어체
3,OOF에 대한 연구는 그 중요성에도 불구하고 다른 개발 재원에 대한 연구에 비해 매...,"Despite its importance, research on OOF is ins...",과학/기술/학술자료,문어체
4,이에 기대효용 이론에 기초하여 투자자들의 고차 적률 위험에 대한 선호를 반영할 수 ...,This led to the need to develop general perfor...,과학/기술/학술자료,문어체


In [7]:
df["super_domain"].value_counts()

super_domain
문어체    7329837
구어체    3219332
혼재     1447188
Name: count, dtype: int64

In [7]:
ds["train"] = ds["train"].rename_column("super_domain", "style")

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'domain', 'style'],
        num_rows: 11996357
    })
})

In [13]:
df = ds["train"].to_pandas()
df.head()

,kor,en,domain,style
0,특정 지역에서는 순위 변동이 크게 나타나고 있는 점에 주목해야 한다.,It should be noted that there are significant ...,과학/기술/학술자료,문어체
1,Post-2020 체제에서는 매5년마다 이전 수준보다 개선된 온실가스 감축 목표를 ...,"Under the Post-2020 system, the ratchet mechan...",과학/기술/학술자료,문어체
2,"회귀 모델은 모두 통계적으로 유의하였고, 분산 팽창 계수(VIF)도 모두 2 내외의...",All regression models were statistically signi...,과학/기술/학술자료,문어체
3,OOF에 대한 연구는 그 중요성에도 불구하고 다른 개발 재원에 대한 연구에 비해 매...,"Despite its importance, research on OOF is ins...",과학/기술/학술자료,문어체
4,이에 기대효용 이론에 기초하여 투자자들의 고차 적률 위험에 대한 선호를 반영할 수 ...,This led to the need to develop general perfor...,과학/기술/학술자료,문어체


In [14]:
df["style"].value_counts()

style
문어체    7329837
구어체    3219332
혼재     1447188
Name: count, dtype: int64

In [15]:
df["domain"].value_counts()

domain
과학/기술/학술자료    3822044
일상/대화         3219332
뉴스/시사         1520939
문화/예술/역사      1447188
법률/행정          726818
의학/보건          720037
특허             359999
금융/경제          180000
Name: count, dtype: int64

In [9]:
# 과학/기술/학술자료 일상/대화 뉴스/시사 문화/예술/역사 법률/행정 의학/보건 특허 금융/경제
def map_domain(d: str) -> int:
    if d == "과학/기술/학술자료":
        return 0
    if d == "일상/대화":
        return 1
    if d == "뉴스/시사":
        return 2
    if d == "문화/예술/역사":
        return 3
    if d == "법률/행정":
        return 4
    if d == "의학/보건":
        return 5
    if d == "특허":
        return 6
    if d == "금융/경제":
        return 7
    
    return 8
    

In [10]:
def map_domain_batch(batch):
    batch["cat"] = [map_domain(d) for d in batch["domain"]]
    return batch

In [11]:
ds = ds.map(map_domain_batch, batched=True)

Map:   0%|          | 0/11996357 [00:00<?, ? examples/s]

In [12]:
ds

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'domain', 'style', 'cat'],
        num_rows: 11996357
    })
})

In [13]:
ds.save_to_disk("en_ko_12m_pp")

Saving the dataset (0/10 shards):   0%|          | 0/11996357 [00:00<?, ? examples/s]

In [14]:
df = ds["train"].to_pandas()
df.head()

,kor,en,domain,style,cat
0,특정 지역에서는 순위 변동이 크게 나타나고 있는 점에 주목해야 한다.,It should be noted that there are significant ...,과학/기술/학술자료,문어체,0
1,Post-2020 체제에서는 매5년마다 이전 수준보다 개선된 온실가스 감축 목표를 ...,"Under the Post-2020 system, the ratchet mechan...",과학/기술/학술자료,문어체,0
2,"회귀 모델은 모두 통계적으로 유의하였고, 분산 팽창 계수(VIF)도 모두 2 내외의...",All regression models were statistically signi...,과학/기술/학술자료,문어체,0
3,OOF에 대한 연구는 그 중요성에도 불구하고 다른 개발 재원에 대한 연구에 비해 매...,"Despite its importance, research on OOF is ins...",과학/기술/학술자료,문어체,0
4,이에 기대효용 이론에 기초하여 투자자들의 고차 적률 위험에 대한 선호를 반영할 수 ...,This led to the need to develop general perfor...,과학/기술/학술자료,문어체,0


In [15]:
df["cat"].value_counts()

cat
0    3822044
1    3219332
2    1520939
3    1447188
4     726818
5     720037
6     359999
7     180000
Name: count, dtype: int64

In [16]:
df["domain"].value_counts()

domain
과학/기술/학술자료    3822044
일상/대화         3219332
뉴스/시사         1520939
문화/예술/역사      1447188
법률/행정          726818
의학/보건          720037
특허             359999
금융/경제          180000
Name: count, dtype: int64

In [17]:
df["style"].value_counts()

style
문어체    7329837
구어체    3219332
혼재     1447188
Name: count, dtype: int64

### 중복 제거 (한국어, 영어 문장 쌍이 같은 경우, 한국어 문장 같고 영어 번역 결과가 다른 경우(학습에 혼란) 제거)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets

import multiprocessing

from functools import partial

from tqdm import tqdm
from collections import Counter

In [2]:
data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_pp"

In [3]:
ds = load_from_disk(data_path)

In [4]:
dataset = ds['train']

In [5]:
df = dataset.to_pandas()

In [12]:
dup_check = df.duplicated(subset="kor")

In [13]:
dup_check.value_counts()

False    10726732
True      1269625
Name: count, dtype: int64

In [16]:
dup_check_en = df.duplicated(subset="en")

In [17]:
dup_check_en.value_counts()

False    10808345
True      1188012
Name: count, dtype: int64

In [18]:
dup_check_kor_en = df.duplicated(subset=["kor","en"])

In [19]:
dup_check_kor_en.value_counts()

False    10970107
True      1026250
Name: count, dtype: int64

In [14]:
dup_check_all = df.duplicated()

In [15]:
dup_check_all.value_counts()

False    11194068
True       802289
Name: count, dtype: int64

In [6]:
df_c = df.drop_duplicates(subset=["kor", "en"])

In [7]:
c_dup_check = df_c.duplicated(subset="kor")
c_dup_check.value_counts()

False    10726732
True       243375
Name: count, dtype: int64

In [23]:
c_dup_check_koren = df_c.duplicated(subset=["kor", "en"])
c_dup_check_koren.value_counts()

False    10970107
Name: count, dtype: int64

In [8]:
df_clean = df_c.drop_duplicates(subset="kor")

In [9]:
clean_dup_check = df_clean.duplicated(subset="kor")
clean_dup_check.value_counts()

False    10726732
Name: count, dtype: int64

In [26]:
clean_dup_check_en = df_clean.duplicated(subset="en")
clean_dup_check_en.value_counts()

False    10583646
True       143086
Name: count, dtype: int64

In [10]:
df_clean.to_csv("data_clean.csv", index=False)

In [11]:
df_clean_en = df_c.drop_duplicates(subset="en")

In [12]:
clean_dup_check_en = df_clean_en.duplicated(subset="en")
clean_dup_check_en.value_counts()

False    10808345
Name: count, dtype: int64

In [30]:
clean_dup_check = df_clean_en.duplicated(subset="kor")
clean_dup_check.value_counts()

False    10585376
True       222969
Name: count, dtype: int64

In [13]:
df_clean_en.to_csv("data_clean_en.csv", index=False)

In [32]:
# ds_clean = DatasetDict(
#     {
#         "train": Dataset.from_pandas(df_clean.reset_index(drop=True))
#     }
# )

: 

### class label 추가

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets

import multiprocessing

from functools import partial

from tqdm import tqdm
from collections import Counter

In [2]:
ds = load_dataset("csv", data_files="data_clean.csv",)

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['kor', 'en', 'domain', 'style', 'cat'],
        num_rows: 10726732
    })
})

In [4]:
dataset = ds["train"]

In [5]:
unique_classes = dataset.unique('style')  
# 문어체    7329837
# 구어체    3219332
# 혼재     1447188

print(f"==>> unique_classes: {unique_classes}")


==>> unique_classes: ['문어체', '혼재', '구어체']


In [6]:
print(f"==>> type(unique_classes): {type(unique_classes)}")

==>> type(unique_classes): <class 'list'>


In [7]:

class_label = ClassLabel(names=['문어체', '구어체', '혼재'])
print(f"==>> class_label: {class_label}")


==>> class_label: ClassLabel(names=['문어체', '구어체', '혼재'])


In [8]:
# def cast_style(example):
#     example["style_class"] = class_label.str2int(example["style"])
#     return example

# dataset = dataset.map(cast_style)

def cast_style(example):
    example["style_class"] = [class_label.str2int(s) for s in example["style"]]
    return example

dataset = dataset.map(cast_style, batched=True)

In [9]:
dataset[:5]

{'kor': ['특정 지역에서는 순위 변동이 크게 나타나고 있는 점에 주목해야 한다.',
  'Post-2020 체제에서는 매5년마다 이전 수준보다 개선된 온실가스 감축 목표를 유엔에 제출해야 하는 래칫 메커니즘이 적용되므로, 국내 최대 온실가스 배출 원인 발전 부문의 감축 부담도 크게 증가할 전망이다.',
  '회귀 모델은 모두 통계적으로 유의하였고, 분산 팽창 계수(VIF)도 모두 2 내외의 양호한 값을 보여 다중 공선성의 문제도 크지 않음을 확인하였다.',
  'OOF에 대한 연구는 그 중요성에도 불구하고 다른 개발 재원에 대한 연구에 비해 매우 부족한 실정이며, 다음의 측면에서 한계를 보이고 있다.',
  '이에 기대효용 이론에 기초하여 투자자들의 고차 적률 위험에 대한 선호를 반영할 수 있는 일반적 성과 지표를 개발할 필요성이 대두되었다.'],
 'en': ['It should be noted that there are significant ranking fluctuations in certain regions.',
  "Under the Post-2020 system, the ratchet mechanism, which requires the U.N. to submit improved greenhouse gas reduction targets every five years, is expected to significantly increase the burden on the nation's largest greenhouse gas emission-causing power generation.",
  'All regression models were statistically significant, and the variance inflation coefficient (VIF) all showed good values around 2, confirming that the problem of multicollinearity was not significa

In [10]:
ds["train"] = dataset

In [11]:
unique_classes = dataset.unique('style_class')
print(f"==>> unique_classes: {unique_classes}")

==>> unique_classes: [0, 2, 1]


In [12]:
class_label = ClassLabel(names=[0, 1, 2])

In [13]:
ds["train"] = ds["train"].cast_column('style_class', class_label)

Casting the dataset:   0%|          | 0/10726732 [00:00<?, ? examples/s]

In [14]:
ds["train"][:5]

{'kor': ['특정 지역에서는 순위 변동이 크게 나타나고 있는 점에 주목해야 한다.',
  'Post-2020 체제에서는 매5년마다 이전 수준보다 개선된 온실가스 감축 목표를 유엔에 제출해야 하는 래칫 메커니즘이 적용되므로, 국내 최대 온실가스 배출 원인 발전 부문의 감축 부담도 크게 증가할 전망이다.',
  '회귀 모델은 모두 통계적으로 유의하였고, 분산 팽창 계수(VIF)도 모두 2 내외의 양호한 값을 보여 다중 공선성의 문제도 크지 않음을 확인하였다.',
  'OOF에 대한 연구는 그 중요성에도 불구하고 다른 개발 재원에 대한 연구에 비해 매우 부족한 실정이며, 다음의 측면에서 한계를 보이고 있다.',
  '이에 기대효용 이론에 기초하여 투자자들의 고차 적률 위험에 대한 선호를 반영할 수 있는 일반적 성과 지표를 개발할 필요성이 대두되었다.'],
 'en': ['It should be noted that there are significant ranking fluctuations in certain regions.',
  "Under the Post-2020 system, the ratchet mechanism, which requires the U.N. to submit improved greenhouse gas reduction targets every five years, is expected to significantly increase the burden on the nation's largest greenhouse gas emission-causing power generation.",
  'All regression models were statistically significant, and the variance inflation coefficient (VIF) all showed good values around 2, confirming that the problem of multicollinearity was not significa

In [15]:
df = ds["train"].to_pandas()

In [16]:
df["style_class"].value_counts()

style_class
0    6541468
1    2923158
2    1262106
Name: count, dtype: int64

In [17]:
ds.save_to_disk("en_ko_12m_ppp")

Saving the dataset (0/9 shards):   0%|          | 0/10726732 [00:00<?, ? examples/s]

## 새 데이터셋에 토크나이저 unk 토큰 개수 확인

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets

import multiprocessing

from functools import partial

from tqdm import tqdm
from collections import Counter

In [2]:
data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_ppp"
max_token_length = 512
batch_size_train = 64
num_workers = 4
batch_size_val = 64
batch_size_test = 64
val_num_workers = 4
start_idx = 64100
end_idx = 1
padding_idx = 0
unk_idx = 2
seed = 42

In [3]:
class Loaders():
    def __init__(
            self,
            data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",
            max_token_length = 512,
            batch_size_train = 8,
            num_workers = 4,
            batch_size_val = 4,
            batch_size_test = 4,
            val_num_workers = 4,
            start_idx = 64100, 
            end_idx = 1, 
            padding_idx = 0, 
            unk_idx = 2,
            seed = 42,
    ):
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.padding_idx = padding_idx
        self.unk_idx = unk_idx

        # 1) 데이터셋 로드
        dataset = load_from_disk(data_path)['train']

        # 4) stratify_by_column으로 분할
        # train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='cat')
        # valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='cat')
        train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='style_class')
        valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='style_class')

        dataset_dict = DatasetDict({
            'train': train_validtest['train'],
            'validation': valid_test['train'],
            'test': valid_test['test']
        })

        # print(dataset_dict)

        NUM_CPU = multiprocessing.cpu_count()
        # print(f"==>> NUM_CPU: {NUM_CPU}")

        # self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
        # @@@ 점포, 만료 등의 단어가 <unk>인 문제 발견 => 다른 토크나이저 사용?
        self.tokenizer = AutoTokenizer.from_pretrained("KETI-AIR/ke-t5-base")

        special_tokens_dict = {'bos_token': '<s>'}
        self.tokenizer.add_special_tokens(special_tokens_dict)

        print(self.tokenizer.all_special_ids)
        print(self.tokenizer.all_special_tokens)

        print(f"==>> self.tokenizer.model_max_length: {self.tokenizer.model_max_length}")

        print(f"==>> len(self.tokenizer): {len(self.tokenizer)}")

        self.max_token_length = min(max_token_length, self.tokenizer.model_max_length)

        # partial을 이용해 tokenizer, max_token_length 인자 고정
        cetf = partial(convert_examples_to_features, tokenizer=self.tokenizer, max_token_length=self.max_token_length)
        # @@@ convert_examples_to_features함수에서 examples가 첫번쨰 인자가 아니면 
        # @@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서 
        # @@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러 발생

        self.datasets = dataset_dict.map(
                                cetf,
                                # lambda examples: convert_examples_to_features(examples, tokenizer=self.tokenizer, max_token_length=self.max_token_length),
                                batched=True,
                                # 원 데이터 'en', 'kor', 'cat' 등의 칼럼을 지우려면
                                # remove_columns 인자 사용
                                # remove_columns=dataset_dict["train"].column_names,
                                num_proc=NUM_CPU)

        print(f"==>> self.datasets: {self.datasets}")

        self.train_set = self.datasets['train']
        self.val_set = self.datasets['validation']
        self.test_set = self.datasets['test']

        c_fn = partial(collate_fn, start_idx=self.start_idx, end_idx=self.end_idx, padding_idx=self.padding_idx, unk_idx=self.unk_idx)

        self.loader_train = DataLoader(self.train_set, batch_size=batch_size_train, collate_fn=c_fn, shuffle=True, num_workers=num_workers, pin_memory=True)
        # 학습시에만 shuffle=True
        self.loader_val = DataLoader(self.val_set, batch_size=batch_size_val, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)
        self.loader_test = DataLoader(self.test_set, batch_size=batch_size_test, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)

# def convert_examples_to_features(tokenizer, max_token_length, examples):
# @@@@@@@@@ examples가 첫번쨰 인자가 아니면 
# @@@@@@@@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서
# @@@@@@@@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러가 발생한다
def convert_examples_to_features(examples, tokenizer, max_token_length):
    # try:
    model_inputs = tokenizer(examples['kor'],
                            text_target=examples['en'],
                            max_length=max_token_length, truncation=True)
    # 여기서 첫번째 인자와 두번째 인자의 순서를 바꾸면 한영 번역 대신 영한 번역용으로 인풋과 타겟이 토큰화된다
    return model_inputs
    # except Exception as e:
    #     # print(f"==>> error examples: {examples}")
    #     raise e


def collate_fn(batch, start_idx, end_idx, padding_idx, unk_idx):
    print('Original:\n', batch)
    # print("".center(50, "-"))
    # batch는 [{'kor':..., 'en':..., 'cat':숫자, 'input_ids':[...], 'attention_mask':[1, ...], 'labels': [...]}, ...] 형태
    
    # keys = batch[0].keys()
    keys = ['kor', 'en', 'domain', 'cat', 'style', 'style_class', 'input_ids', 'attention_mask', 'labels']
    # print(f"==>> keys: {keys}")
    # print("".center(50, "-"))
    
    # new_batch = {k:[] for k in keys}
    # new_batch['decoder_inputs'] = []
    
    # for b in batch:
    #     for k,v in b.items():
    #         if k == 'input_ids' or k == 'attention_mask':
    #             new_batch[k].append(torch.LongTensor(v))
    #         elif k == 'labels':
    #             new_batch[k].append(torch.LongTensor(v))
    #             new_batch['decoder_inputs'].append(torch.LongTensor([65001] + v[:-1]))
    #         else:
    #             new_batch[k].append(v)

    # key값별로 value 다 모으기
    new_batch = {k:[b[k] for b in batch] for k in keys}

    # list들 LongTensor로 변환
    new_batch['decoder_inputs'] = [torch.LongTensor([start_idx] + label[:-1]) for label in new_batch['labels']]
    # print(f"==>> new_batch['decoder_inputs']: {new_batch['decoder_inputs']}")
    new_batch['input_ids'] = [torch.LongTensor(inp) for inp in new_batch['input_ids']]
    new_batch['labels'] = [torch.LongTensor(label) for label in new_batch['labels']]
    new_batch['attention_mask'] = [torch.LongTensor(mask) for mask in new_batch['attention_mask']]

    new_batch['ntokens'] = sum([l.numel() for l in new_batch['labels']])

    # decoder input의 attention mask 생성
    new_batch['decoder_mask'] = [torch.ones_like(d_inp, dtype=torch.long) for d_inp in new_batch['decoder_inputs']]


    # 각 input과 target 텐서를 패딩
    padded_inputs = pad_sequence(new_batch['input_ids'], batch_first=True, padding_value=padding_idx)
    new_batch['input_ids'] = padded_inputs
    padded_decoder_inputs = pad_sequence(new_batch['decoder_inputs'], batch_first=True, padding_value=padding_idx)
    new_batch['decoder_inputs'] = padded_decoder_inputs
    padded_targets = pad_sequence(new_batch['labels'], batch_first=True, padding_value=padding_idx)
    new_batch['labels'] = padded_targets

    # attention 마스크는 패딩(padding_idx) 대신 False(0)을 입력
    padded_masks = pad_sequence(new_batch['attention_mask'], batch_first=True, padding_value=0)
    new_batch['attention_mask'] = padded_masks

    padded_d_masks = pad_sequence(new_batch['decoder_mask'], batch_first=True, padding_value=0)
    new_batch['decoder_mask'] = padded_d_masks
    
    return new_batch

In [ ]:
loaders = Loaders(
    data_path=data_path,
    max_token_length=max_token_length,
    batch_size_train=batch_size_train,
    num_workers=num_workers,
    batch_size_val=batch_size_val,
    batch_size_test=batch_size_test,
    val_num_workers=val_num_workers,
    start_idx=start_idx,
    end_idx=end_idx,
    padding_idx=padding_idx,
    unk_idx=unk_idx,
    seed=seed,
)

In [ ]:
print(f"==>> unk_idx: {unk_idx}")

input_total = Counter([])
target_total = Counter([])
# total

num_batches_train = len(loaders.loader_train)

for step, batch in tqdm(
    enumerate(loaders.loader_train),
    total=num_batches_train,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total += input_count
    target_total += target_count

    # if step == 10:
    #     break

print(f"==>> input_total: {input_total}")
print(f"==>> target_total: {target_total}")

print(f"==>> input_total[unk_idx]: {input_total[unk_idx]}")
print(f"==>> target_total[unk_idx]: {target_total[unk_idx]}")

input_total_val = Counter([])
target_total_val = Counter([])
# total

num_batches_val = len(loaders.loader_val)

for step, batch in tqdm(
    enumerate(loaders.loader_val),
    total=num_batches_val,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_val += input_count
    target_total_val += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_val: {input_total_val}")
print(f"==>> target_total_val: {target_total_val}")
print(f"==>> input_total_val[unk_idx]: {input_total_val[unk_idx]}")
print(f"==>> target_total_val[unk_idx]: {target_total_val[unk_idx]}")

input_total_test = Counter([])
target_total_test = Counter([])
# total

num_batches_test = len(loaders.loader_test)

for step, batch in tqdm(
    enumerate(loaders.loader_test),
    total=num_batches_test,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_test += input_count
    target_total_test += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_test: {input_total_test}")
print(f"==>> target_total_test: {target_total_test}")
print(f"==>> input_total_test[unk_idx]: {input_total_test[unk_idx]}")
print(f"==>> target_total_test[unk_idx]: {target_total_test[unk_idx]}")

In [ ]:
# tokenizer_name = "LGAI-EXAONE/K-EXAONE-236B-A23B"

data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_ppp"
max_token_length = 512
batch_size_train = 64
num_workers = 4
batch_size_val = 64
batch_size_test = 64
val_num_workers = 4
start_idx = 1
end_idx = 53
padding_idx = 0
unk_idx = 3
seed = 42

In [ ]:
class LoadersLGEXA236():
    def __init__(
            self,
            data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",
            max_token_length = 512,
            batch_size_train = 8,
            num_workers = 4,
            batch_size_val = 4,
            batch_size_test = 4,
            val_num_workers = 4,
            start_idx = 64100, 
            end_idx = 1, 
            padding_idx = 0, 
            unk_idx = 2,
            seed = 42,
    ):
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.padding_idx = padding_idx
        self.unk_idx = unk_idx

        # 1) 데이터셋 로드
        dataset = load_from_disk(data_path)['train']
        
        # 4) stratify_by_column으로 분할
        # train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='cat')
        # valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='cat')
        train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='style_class')
        valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='style_class')

        dataset_dict = DatasetDict({
            'train': train_validtest['train'],
            'validation': valid_test['train'],
            'test': valid_test['test']
        })

        # print(dataset_dict)

        NUM_CPU = multiprocessing.cpu_count()
        # print(f"==>> NUM_CPU: {NUM_CPU}")

        # self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
        # @@@ 점포, 만료 등의 단어가 <unk>인 문제 발견 => 다른 토크나이저 사용?
        self.tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/K-EXAONE-236B-A23B")

        print(self.tokenizer.all_special_ids)
        print(self.tokenizer.all_special_tokens)

        print(f"==>> self.tokenizer.model_max_length: {self.tokenizer.model_max_length}")

        print(f"==>> len(self.tokenizer): {len(self.tokenizer)}")

        self.max_token_length = min(max_token_length, self.tokenizer.model_max_length)
        print(f"==>> self.max_token_length: {self.max_token_length}")

        # partial을 이용해 tokenizer, max_token_length 인자 고정
        cetf = partial(convert_examples_to_features, tokenizer=self.tokenizer, max_token_length=self.max_token_length)
        # @@@ convert_examples_to_features함수에서 examples가 첫번쨰 인자가 아니면 
        # @@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서 
        # @@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러 발생

        self.datasets = dataset_dict.map(
                                cetf,
                                # lambda examples: convert_examples_to_features(examples, tokenizer=self.tokenizer, max_token_length=self.max_token_length),
                                batched=True,
                                # 원 데이터 'en', 'kor', 'cat' 등의 칼럼을 지우려면
                                # remove_columns 인자 사용
                                # remove_columns=dataset_dict["train"].column_names,
                                num_proc=NUM_CPU)

        print(f"==>> self.datasets: {self.datasets}")

        self.train_set = self.datasets['train']
        self.val_set = self.datasets['validation']
        self.test_set = self.datasets['test']

        c_fn = partial(collate_fn, start_idx=self.start_idx, end_idx=self.end_idx, padding_idx=self.padding_idx, unk_idx=self.unk_idx)

        self.loader_train = DataLoader(self.train_set, batch_size=batch_size_train, collate_fn=c_fn, shuffle=True, num_workers=num_workers, pin_memory=True)
        # 학습시에만 shuffle=True
        self.loader_val = DataLoader(self.val_set, batch_size=batch_size_val, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)
        self.loader_test = DataLoader(self.test_set, batch_size=batch_size_test, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)

# def convert_examples_to_features(tokenizer, max_token_length, examples):
# @@@@@@@@@ examples가 첫번쨰 인자가 아니면 
# @@@@@@@@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서
# @@@@@@@@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러가 발생한다
def convert_examples_to_features(examples, tokenizer, max_token_length):
    model_inputs = tokenizer(examples['kor'],
                             text_target=examples['en'],
                             max_length=max_token_length, truncation=True)
    # 여기서 첫번째 인자와 두번째 인자의 순서를 바꾸면 한영 번역 대신 영한 번역용으로 인풋과 타겟이 토큰화된다
    return model_inputs


def collate_fn(batch, start_idx, end_idx, padding_idx, unk_idx):
    # print('Original:\n', batch)
    # print("".center(50, "-"))
    # batch는 [{'kor':..., 'en':..., 'cat':숫자, 'input_ids':[...], 'attention_mask':[1, ...], 'labels': [...]}, ...] 형태
    
    # keys = batch[0].keys()
    keys = ['kor', 'en', 'domain', 'cat', 'style', 'style_class', 'input_ids', 'attention_mask', 'labels']
    # print(f"==>> keys: {keys}")
    # print("".center(50, "-"))
    
    # new_batch = {k:[] for k in keys}
    # new_batch['decoder_inputs'] = []
    
    # for b in batch:
    #     for k,v in b.items():
    #         if k == 'input_ids' or k == 'attention_mask':
    #             new_batch[k].append(torch.LongTensor(v))
    #         elif k == 'labels':
    #             new_batch[k].append(torch.LongTensor(v))
    #             new_batch['decoder_inputs'].append(torch.LongTensor([65001] + v[:-1]))
    #         else:
    #             new_batch[k].append(v)

    # key값별로 value 다 모으기
    new_batch = {k:[b[k] for b in batch] for k in keys}

    # list들 LongTensor로 변환
    new_batch['decoder_inputs'] = [torch.LongTensor([start_idx] + label[:-1]) for label in new_batch['labels']]
    # print(f"==>> new_batch['decoder_inputs']: {new_batch['decoder_inputs']}")
    new_batch['input_ids'] = [torch.LongTensor(inp) for inp in new_batch['input_ids']]
    new_batch['labels'] = [torch.LongTensor(label) for label in new_batch['labels']]
    new_batch['attention_mask'] = [torch.LongTensor(mask) for mask in new_batch['attention_mask']]

    new_batch['ntokens'] = sum([l.numel() for l in new_batch['labels']])

    # decoder input의 attention mask 생성
    new_batch['decoder_mask'] = [torch.ones_like(d_inp, dtype=torch.long) for d_inp in new_batch['decoder_inputs']]


    # 각 input과 target 텐서를 패딩
    padded_inputs = pad_sequence(new_batch['input_ids'], batch_first=True, padding_value=padding_idx)
    new_batch['input_ids'] = padded_inputs
    padded_decoder_inputs = pad_sequence(new_batch['decoder_inputs'], batch_first=True, padding_value=padding_idx)
    new_batch['decoder_inputs'] = padded_decoder_inputs
    padded_targets = pad_sequence(new_batch['labels'], batch_first=True, padding_value=padding_idx)
    new_batch['labels'] = padded_targets

    # attention 마스크는 패딩(padding_idx) 대신 False(0)을 입력
    padded_masks = pad_sequence(new_batch['attention_mask'], batch_first=True, padding_value=0)
    new_batch['attention_mask'] = padded_masks

    padded_d_masks = pad_sequence(new_batch['decoder_mask'], batch_first=True, padding_value=0)
    new_batch['decoder_mask'] = padded_d_masks
    
    return new_batch

In [ ]:
loaders = LoadersLGEXA236(
    data_path=data_path,
    max_token_length=max_token_length,
    batch_size_train=batch_size_train,
    num_workers=num_workers,
    batch_size_val=batch_size_val,
    batch_size_test=batch_size_test,
    val_num_workers=val_num_workers,
    start_idx=start_idx,
    end_idx=end_idx,
    padding_idx=padding_idx,
    unk_idx=unk_idx,
    seed=seed,
)

In [ ]:
print(f"==>> unk_idx: {unk_idx}")

input_total = Counter([])
target_total = Counter([])
# total

num_batches_train = len(loaders.loader_train)

for step, batch in tqdm(
    enumerate(loaders.loader_train),
    total=num_batches_train,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total += input_count
    target_total += target_count

    # if step == 10:
    #     break

print(f"==>> input_total: {input_total}")
print(f"==>> target_total: {target_total}")
print(f"==>> input_total[unk_idx]: {input_total[unk_idx]}")
print(f"==>> target_total[unk_idx]: {target_total[unk_idx]}")


input_total_val = Counter([])
target_total_val = Counter([])
# total

num_batches_val = len(loaders.loader_val)

for step, batch in tqdm(
    enumerate(loaders.loader_val),
    total=num_batches_val,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_val += input_count
    target_total_val += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_val: {input_total_val}")
print(f"==>> target_total_val: {target_total_val}")
print(f"==>> input_total_val[unk_idx]: {input_total_val[unk_idx]}")
print(f"==>> target_total_val[unk_idx]: {target_total_val[unk_idx]}")


input_total_test = Counter([])
target_total_test = Counter([])
# total

num_batches_test = len(loaders.loader_test)

for step, batch in tqdm(
    enumerate(loaders.loader_test),
    total=num_batches_test,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_test += input_count
    target_total_test += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_test: {input_total_test}")
print(f"==>> target_total_test: {target_total_test}")
print(f"==>> input_total_test[unk_idx]: {input_total_test[unk_idx]}")
print(f"==>> target_total_test[unk_idx]: {target_total_test[unk_idx]}")

In [ ]:
# tokenizer_name = "Translation-EnKo/exaone3-instrucTrans-v2-enko-7.8b"

data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/transformer_experiments/eda/en_ko_12m_ppp"
max_token_length = 512
batch_size_train = 64
num_workers = 4
batch_size_val = 64
batch_size_test = 64
val_num_workers = 4
start_idx = 1
end_idx = 361
padding_idx = 0
unk_idx = 3
seed = 42

In [ ]:
class LoadersLGEXAENKO():
    def __init__(
            self,
            data_path="/home/paokimsiwoong/workspace/github.com/paokimsiwoong/ml_practice/transformer/data.csv",
            max_token_length = 512,
            batch_size_train = 8,
            num_workers = 4,
            batch_size_val = 4,
            batch_size_test = 4,
            val_num_workers = 4,
            start_idx = 64100, 
            end_idx = 1, 
            padding_idx = 0, 
            unk_idx = 2,
            seed = 42,
    ):
        self.start_idx = start_idx
        self.end_idx = end_idx
        self.padding_idx = padding_idx
        self.unk_idx = unk_idx

        # 1) 데이터셋 로드
        dataset = load_from_disk(data_path)['train']
        

        # 4) stratify_by_column으로 분할
        # train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='cat')
        # valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='cat')
        train_validtest = dataset.train_test_split(test_size=0.2, seed=seed, stratify_by_column='style_class')
        valid_test = train_validtest['test'].train_test_split(test_size=0.1, seed=seed, stratify_by_column='style_class')

        dataset_dict = DatasetDict({
            'train': train_validtest['train'],
            'validation': valid_test['train'],
            'test': valid_test['test']
        })

        # print(dataset_dict)

        NUM_CPU = multiprocessing.cpu_count()
        # print(f"==>> NUM_CPU: {NUM_CPU}")

        # self.tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
        # @@@ 점포, 만료 등의 단어가 <unk>인 문제 발견 => 다른 토크나이저 사용?
        self.tokenizer = AutoTokenizer.from_pretrained("Translation-EnKo/exaone3-instrucTrans-v2-enko-7.8b")

        special_tokens_dict = {'pad_token': '[PAD]'}
        self.tokenizer.add_special_tokens(special_tokens_dict)
        
        print(self.tokenizer.all_special_ids)
        print(self.tokenizer.all_special_tokens)

        print(f"==>> self.tokenizer.model_max_length: {self.tokenizer.model_max_length}")

        print(f"==>> len(self.tokenizer): {len(self.tokenizer)}")

        self.max_token_length = min(max_token_length, self.tokenizer.model_max_length)

        # partial을 이용해 tokenizer, max_token_length 인자 고정
        cetf = partial(convert_examples_to_features, tokenizer=self.tokenizer, max_token_length=self.max_token_length)
        # @@@ convert_examples_to_features함수에서 examples가 첫번쨰 인자가 아니면 
        # @@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서 
        # @@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러 발생

        self.datasets = dataset_dict.map(
                                cetf,
                                # lambda examples: convert_examples_to_features(examples, tokenizer=self.tokenizer, max_token_length=self.max_token_length),
                                batched=True,
                                # 원 데이터 'en', 'kor', 'cat' 등의 칼럼을 지우려면
                                # remove_columns 인자 사용
                                # remove_columns=dataset_dict["train"].column_names,
                                num_proc=NUM_CPU)

        print(f"==>> self.datasets: {self.datasets}")

        self.train_set = self.datasets['train']
        self.val_set = self.datasets['validation']
        self.test_set = self.datasets['test']

        c_fn = partial(collate_fn, start_idx=self.start_idx, end_idx=self.end_idx, padding_idx=self.padding_idx, unk_idx=self.unk_idx)

        self.loader_train = DataLoader(self.train_set, batch_size=batch_size_train, collate_fn=c_fn, shuffle=True, num_workers=num_workers, pin_memory=True)
        # 학습시에만 shuffle=True
        self.loader_val = DataLoader(self.val_set, batch_size=batch_size_val, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)
        self.loader_test = DataLoader(self.test_set, batch_size=batch_size_test, collate_fn=c_fn, shuffle=False, num_workers=val_num_workers, pin_memory=True)

# def convert_examples_to_features(tokenizer, max_token_length, examples):
# @@@@@@@@@ examples가 첫번쨰 인자가 아니면 
# @@@@@@@@@ partial과 dataset_dict.map의 인자 전달 방식이 충돌해서
# @@@@@@@@@ TypeError: convert_examples_to_features() got multiple values for argument 'tokenizer' 에러가 발생한다
def convert_examples_to_features(examples, tokenizer, max_token_length):
    model_inputs = tokenizer(examples['kor'],
                             text_target=examples['en'],
                             max_length=max_token_length, truncation=True)
    # 여기서 첫번째 인자와 두번째 인자의 순서를 바꾸면 한영 번역 대신 영한 번역용으로 인풋과 타겟이 토큰화된다
    return model_inputs


def collate_fn(batch, start_idx, end_idx, padding_idx, unk_idx):
    # print('Original:\n', batch)
    # print("".center(50, "-"))
    # batch는 [{'kor':..., 'en':..., 'cat':숫자, 'input_ids':[...], 'attention_mask':[1, ...], 'labels': [...]}, ...] 형태
    
    # keys = batch[0].keys()
    keys = ['kor', 'en', 'domain', 'cat', 'style', "style_class", 'input_ids', 'attention_mask', 'labels']
    # print(f"==>> keys: {keys}")
    # print("".center(50, "-"))
    
    # new_batch = {k:[] for k in keys}
    # new_batch['decoder_inputs'] = []
    
    # for b in batch:
    #     for k,v in b.items():
    #         if k == 'input_ids' or k == 'attention_mask':
    #             new_batch[k].append(torch.LongTensor(v))
    #         elif k == 'labels':
    #             new_batch[k].append(torch.LongTensor(v))
    #             new_batch['decoder_inputs'].append(torch.LongTensor([65001] + v[:-1]))
    #         else:
    #             new_batch[k].append(v)

    # key값별로 value 다 모으기
    new_batch = {k:[b[k] for b in batch] for k in keys}

    # list들 LongTensor로 변환
    new_batch['decoder_inputs'] = [torch.LongTensor([start_idx] + label[:-1]) for label in new_batch['labels']]
    # print(f"==>> new_batch['decoder_inputs']: {new_batch['decoder_inputs']}")
    new_batch['input_ids'] = [torch.LongTensor(inp) for inp in new_batch['input_ids']]
    new_batch['labels'] = [torch.LongTensor(label) for label in new_batch['labels']]
    new_batch['attention_mask'] = [torch.LongTensor(mask) for mask in new_batch['attention_mask']]

    new_batch['ntokens'] = sum([l.numel() for l in new_batch['labels']])

    # decoder input의 attention mask 생성
    new_batch['decoder_mask'] = [torch.ones_like(d_inp, dtype=torch.long) for d_inp in new_batch['decoder_inputs']]


    # 각 input과 target 텐서를 패딩
    padded_inputs = pad_sequence(new_batch['input_ids'], batch_first=True, padding_value=padding_idx)
    new_batch['input_ids'] = padded_inputs
    padded_decoder_inputs = pad_sequence(new_batch['decoder_inputs'], batch_first=True, padding_value=padding_idx)
    new_batch['decoder_inputs'] = padded_decoder_inputs
    padded_targets = pad_sequence(new_batch['labels'], batch_first=True, padding_value=padding_idx)
    new_batch['labels'] = padded_targets

    # attention 마스크는 패딩(padding_idx) 대신 False(0)을 입력
    padded_masks = pad_sequence(new_batch['attention_mask'], batch_first=True, padding_value=0)
    new_batch['attention_mask'] = padded_masks

    padded_d_masks = pad_sequence(new_batch['decoder_mask'], batch_first=True, padding_value=0)
    new_batch['decoder_mask'] = padded_d_masks
    
    return new_batch

In [ ]:
loaders = LoadersLGEXAENKO(
    data_path=data_path,
    max_token_length=max_token_length,
    batch_size_train=batch_size_train,
    num_workers=num_workers,
    batch_size_val=batch_size_val,
    batch_size_test=batch_size_test,
    val_num_workers=val_num_workers,
    start_idx=start_idx,
    end_idx=end_idx,
    padding_idx=padding_idx,
    unk_idx=unk_idx,
    seed=seed,
)

In [ ]:
print(f"==>> unk_idx: {unk_idx}")

input_total = Counter([])
target_total = Counter([])
# total

num_batches_train = len(loaders.loader_train)

for step, batch in tqdm(
    enumerate(loaders.loader_train),
    total=num_batches_train,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total += input_count
    target_total += target_count

    # if step == 10:
    #     break

print(f"==>> input_total: {input_total}")
print(f"==>> target_total: {target_total}")
print(f"==>> input_total[unk_idx]: {input_total[unk_idx]}")
print(f"==>> target_total[unk_idx]: {target_total[unk_idx]}")

input_total_val = Counter([])
target_total_val = Counter([])
# total

num_batches_val = len(loaders.loader_val)

for step, batch in tqdm(
    enumerate(loaders.loader_val),
    total=num_batches_val,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_val += input_count
    target_total_val += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_val: {input_total_val}")
print(f"==>> target_total_val: {target_total_val}")
print(f"==>> input_total_val[unk_idx]: {input_total_val[unk_idx]}")
print(f"==>> target_total_val[unk_idx]: {target_total_val[unk_idx]}")


input_total_test = Counter([])
target_total_test = Counter([])
# total

num_batches_test = len(loaders.loader_test)

for step, batch in tqdm(
    enumerate(loaders.loader_test),
    total=num_batches_test,
):  
    input_count = Counter(batch['input_ids'].view(-1).tolist())
    # print(input_count)
    # print(batch['input_ids'].view(-1).tolist())

    target_count = Counter(batch["labels"].view(-1).tolist())

    input_total_test += input_count
    target_total_test += target_count

    # if step == 10:
    #     break

print(f"==>> input_total_test: {input_total_test}")
print(f"==>> target_total_test: {target_total_test}")
print(f"==>> input_total_test[unk_idx]: {input_total_test[unk_idx]}")
print(f"==>> target_total_test[unk_idx]: {target_total_test[unk_idx]}")

In [ ]:
### TODO: 중복데이터 확인하고 지우기, cat 칼럼 지우기

In [ ]:
# from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict, Dataset, concatenate_datasets
# import pandas as pd

In [ ]:
# dup_check = df.duplicated(subset="target_text")